# Organized MODFLOW Input-Preparation Notebook

This notebook has been reorganized into a cleaner workflow for building **MODFLOW input files**.  
The main goal here is to keep the original logic, but place the steps in a more sensible order and add short explanations before each code block.

## How this notebook is organized
1. Common helpers
2. DEM and lake polygons
3. Water mask and constant-head boundary prep
4. Hydraulic conductivity and model bottom
5. Active-domain raster
6. Recharge processing
7. Streams and drains
8. Great Lakes bathymetry and modified model top
9. Starting heads from wells and hydrography
10. Optional / archived alternative workflows

## Notes
- I preserved the original code as much as possible.
- A few exploratory or duplicate workflows were moved to the end instead of deleting them.
- Some cells still use hard-coded file paths, so update those paths before running on a new machine.
- If you want this notebook to be fully reproducible from top to bottom, the next cleanup step would be to move repeated imports and paths into a small configuration block.


## Optional common imports

This cell gathers a few imports that are used repeatedly across the notebook.  
Most later cells still keep their own imports so they can be run independently, but this block makes the notebook easier to understand at the top.


In [1]:
import os
import glob
import re
import time
from pathlib import Path

import numpy as n
import matplotlib.pyplot as plt
import xarray as xr
import arcpy 
from arcpy.sa import *
import rasterio as rio
from rasterio.merge import merge
from rasterio.warp import calculate_default_transform, reproject, Resampling

## 1. Common helpers
These helpers are reused in later raster and vector preprocessing steps.


### Helper function for raster alignment

Defines utility functions used later to warp rasters to a template grid, clean continuous arrays, and safely add fields to vector data. Keep this near the top because later preprocessing steps reuse these helpers.


In [2]:
def warp_raster_to_template(src_path, template_path, out_path, resampling, dst_nodata=-9999.0):
    with rio.open(template_path) as tmpl, rio.open(src_path) as src:
        dst_meta = src.meta.copy()
        dst_meta.update(
            driver="GTiff",
            crs=tmpl.crs,
            transform=tmpl.transform,
            width=tmpl.width,
            height=tmpl.height,
            nodata=dst_nodata,
            compress="deflate",
            tiled=True,
            BIGTIFF="YES",
            blockxsize=256,
            blockysize=256,
        )
        with rio.open(out_path, "w", **dst_meta) as dst:
            for b in range(1, src.count + 1):
                reproject(
                    source=rio.band(src, b),
                    destination=rio.band(dst, b),
                    src_transform=src.transform,
                    src_crs=src.crs,
                    src_nodata=src.nodata,
                    dst_transform=tmpl.transform,
                    dst_crs=tmpl.crs,
                    dst_nodata=dst_nodata,
                    resampling=resampling,
                )
    return out_path

def clean_continuous(a, nodata, fill=0.0):
    a = a.astype("float32", copy=False)
    a = np.nan_to_num(a, nan=fill, posinf=fill, neginf=fill)
    if nodata is not None:
        a = np.where(a == nodata, fill, a)
    return a

import arcpy

def ensure_field(dataset, name, ftype, length=None):
    """Add a field if it doesn't already exist."""
    existing = {f.name for f in arcpy.ListFields(dataset)}
    if name not in existing:
        if ftype.upper() == "TEXT" and length is not None:
            arcpy.management.AddField(dataset, name, ftype, field_length=length)
        else:
            arcpy.management.AddField(dataset, name, ftype)


## 2. DEM and Great Lakes base layers
Build the base elevation raster and merged lake polygon layer used by later boundary and masking steps.


### Mosaic and reproject DEM tiles

Reads all DEM GeoTIFF tiles, reprojects them to EPSG:3174 at 30 m resolution, and merges them into a single model-top DEM. This creates the base elevation raster used by several later steps.


In [ ]:
import os, glob
import rasterio
from rasterio.merge import merge
from rasterio.warp import calculate_default_transform, reproject, Resampling

Modeltop = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\DEM"
target_crs = "EPSG:3174"
target_res = 30
nodata_out = -9999.0

tmp_dir = os.path.join(Modeltop, "_tmp_reproj_3174")
os.makedirs(tmp_dir, exist_ok=True)

tifs = sorted(glob.glob(os.path.join(Modeltop, "*.tif")))
if not tifs:
    raise FileNotFoundError(f"No .tif files found in: {Modeltop}")

reproj_paths = []

for fp in tifs:
    with rasterio.open(fp) as src:
        if src.crs is None:
            raise ValueError(f"{os.path.basename(fp)} has no CRS. Define Projection first.")

        src_nodata = src.nodata
        out_fp = os.path.join(tmp_dir, os.path.splitext(os.path.basename(fp))[0] + "_3174.tif")

        if not os.path.exists(out_fp):
            transform, width, height = calculate_default_transform(
                src.crs, target_crs, src.width, src.height, *src.bounds
            )

            meta = src.meta.copy()
            meta.update({
                "crs": target_crs,
                "transform": transform,
                "width": width,
                "height": height,
                "dtype": "float32",
                "nodata": nodata_out,
                "compress": "lzw",
                "tiled": True,
                "BIGTIFF": "IF_SAFER"
            })

            with rasterio.open(out_fp, "w", **meta) as dst:
                reproject(
                    source=rasterio.band(src, 1),
                    destination=rasterio.band(dst, 1),
                    src_transform=src.transform,
                    src_crs=src.crs,
                    src_nodata=src_nodata,
                    dst_transform=transform,
                    dst_crs=target_crs,
                    dst_nodata=nodata_out,
                    resampling=Resampling.bilinear
                )

        reproj_paths.append(out_fp)

# Merge and write directly to disk (BigTIFF)
out_tif = os.path.join(Modeltop, "DEM_merged_3174_30m.tif")

srcs = [rasterio.open(fp) for fp in reproj_paths]
template = srcs[0].meta.copy()
template.update({
    "driver": "GTiff",
    "crs": target_crs,
    "dtype": "float32",
    "nodata": nodata_out,
    "compress": "lzw",
    "tiled": True,
    "BIGTIFF": "YES"
})

merge(
    srcs,
    res=(target_res, target_res),
    nodata=nodata_out,
    resampling=Resampling.bilinear,
    target_aligned_pixels=True,
    dst_path=out_tif,
    dst_kwds=template
)

for s in srcs:
    s.close()

print("✅ Wrote:", out_tif)


### Build statistics for the merged DEM

Calculates raster statistics and pyramids for the merged DEM so it displays and processes more reliably in ArcGIS.


In [ ]:
import arcpy
arcpy.env.overwriteOutput = True

out_tif = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\DEM\DEM_merged_3174_30m.tif"

arcpy.management.CalculateStatistics(out_tif)
arcpy.management.BuildPyramids(out_tif)
print("✅ Built statistics + pyramids")

In [12]:
import arcpy

# Clip the DEM to model domain 
out_tif = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\DEM\DEM_merged_3174_30m.tif"

model_domain = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\Ibound\extended_Bdry_final_GLB_Albers_exported.shp"

# create a 20 km buffer around the model domain
buffered_domain = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\Ibound\extended_Bdry_final_GLB_Albers_buffered_20km.shp"

arcpy.analysis.Buffer(model_domain, buffered_domain, "20 Kilometers", dissolve_option="ALL")
print("✅ Created 20 km buffer around model domain")


out_DEM_clipped = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\DEM\DEM_merged_3174_30m_clipped.tif"

✅ Created 20 km buffer around model domain


In [13]:
arcpy.management.Clip(
    in_raster=out_tif,
    rectangle="#",
    out_raster=out_DEM_clipped,
    in_template_dataset=buffered_domain,
    nodata_value="",
    clipping_geometry="ClippingGeometry",
    maintain_clipping_extent="NO_MAINTAIN_EXTENT"
)

<Result 'D:\\Users\\abolmaal\\modelling\\Modflow\\Prep\\GreatLakes\\model_Layers\\DEM\\DEM_merged_3174_30m_clipped.tif'>

### Merge Great Lakes shoreline polygons

Combines all lake polygon shapefiles into one merged Great Lakes polygon layer. This merged hydro layer is later rasterized to make the lake/water mask.


In [ ]:
import os
import arcpy

Lakes_path = r"D:\Users\abolmaal\Arcgis\GreatLakes"
out_shp = os.path.join(Lakes_path, "hydro_p_GreatLakes.shp")

arcpy.env.workspace = Lakes_path
arcpy.env.overwriteOutput = True

# Get all shapefiles in the folder
shps = arcpy.ListFeatureClasses(feature_type="All")
shps = [os.path.join(Lakes_path, s) for s in shps if s.lower().endswith(".shp")]

if not shps:
    raise FileNotFoundError(f"No shapefiles found in: {Lakes_path}")

# Merge
arcpy.management.Merge(shps, out_shp)

print(f"✅ Merged {len(shps)} shapefiles -> {out_shp}")


### Convert the DEM to 1000 meter cell size

In [ ]:
import arcpy

arcpy.env.overwriteOutput = True

# ---------------------------
# INPUT / OUTPUT
# ---------------------------
input_raster = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\DEM\DEM_merged_3174_30m_clipped.tif"
output_raster = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\DEM\DEM_extended20kmbdr_1000m.tif"

target_resolution = 1000  # meters


arcpy.management.Resample(
    in_raster=input_raster,
    out_raster=output_raster,
    cell_size="1000 1000",
    resampling_type="BILINEAR"  # or "AVERAGE" if available
)

print("Done!")

Done!


## 3.Creat an Ibound 
DEM is the 20 km buffered from the main boundary now we want to create an ibound that is getting 1 inside the Great lakes boundary and gets 0 out the the boundary


### Create the Idomain

Rasterizes the Great Lakes polygons onto the DEM-aligned grid and builds a domain water mask. This mask is the starting point for defining lake cells, land cells, and shoreline CHD candidates.


In [15]:
import os
import arcpy
from arcpy import sa

arcpy.CheckOutExtension("Spatial")
arcpy.env.overwriteOutput = True

# ----------------------------
# INPUTS
# ----------------------------
dem = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\DEM\DEM_merged_3174_30m_clipped.tif"

inner_boundary = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\Ibound\extended_Bdry_final_GLB_Albers_exported.shp"
outer_boundary = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\Ibound\extended_Bdry_final_GLB_Albers_buffered_20km.shp"
out_dir = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\Ibound"
os.makedirs(out_dir, exist_ok=True)

# ----------------------------
# ENV ALIGNMENT
# ----------------------------
dem_r  = arcpy.Raster(dem)
dem_sr = arcpy.Describe(dem).spatialReference

arcpy.env.snapRaster = dem
arcpy.env.cellSize   = dem
arcpy.env.outputCoordinateSystem = dem_sr

# KEY CHANGE: set extent to the boundary shapefile, not the DEM
arcpy.env.extent = outer_boundary

print("DEM rows/cols:", dem_r.height, dem_r.width)
print("DEM cellsize:", dem_r.meanCellWidth, dem_r.meanCellHeight)
print("DEM SR:", dem_sr.name, "factoryCode:", dem_sr.factoryCode)
print("Output extent set to:", outer_boundary)

# ----------------------------
# HELPERS
# ----------------------------
def safe_delete(path):
    if arcpy.Exists(path):
        arcpy.management.Delete(path)

def project_if_needed(fc, out_name):
    sr = arcpy.Describe(fc).spatialReference
    if sr.factoryCode != dem_sr.factoryCode:
        out_fc = os.path.join(out_dir, out_name)
        safe_delete(out_fc)
        arcpy.management.Project(fc, out_fc, dem_sr)
        return out_fc
    return fc

def ensure_constant_field(fc, field_name, value=1):
    fset = {f.name.upper() for f in arcpy.ListFields(fc)}
    if field_name.upper() not in fset:
        arcpy.management.AddField(fc, field_name, "SHORT")
    arcpy.management.CalculateField(fc, field_name, str(value), "PYTHON3")
    return field_name

# ----------------------------
# PROJECT BOUNDARY IF NEEDED
# ----------------------------
bnd_use = project_if_needed(inner_boundary, "boundary_proj.shp")

try:
    arcpy.management.RepairGeometry(bnd_use)
except Exception as e:
    print("⚠️ RepairGeometry skipped/failed:", e)

# ----------------------------
# RASTERIZE BOUNDARY (no buffer, no lake masking)
#   Inside boundary = 1
#   Outside boundary = 0
# ----------------------------
bnd_field = ensure_constant_field(bnd_use, "BNDVAL", 1)

bnd_raw = os.path.join(out_dir, "boundary_raw.tif")
safe_delete(bnd_raw)


# final = sa.Int(sa.Con(sa.IsNull(bnd_raw), 0, 1))
# Step 1 – create a 1-valued raster aligned to DEM grid within bbox
const_1 = sa.CreateConstantRaster(1, "INTEGER", dem)
inside_inner = sa.ExtractByMask(const_1, inner_boundary)
# Step 2 – Fill between inner and outer boundary with 0
domain = sa.Int(sa.Con(sa.IsNull(inside_inner), 0, inside_inner))

# Step 3 – clip by outer boundary --> remove the rectangulare conrners outside the buffer
final = sa.ExtractByMask(domain, outer_boundary)
# end main

# ----------------------------
# SAVE TO FILE GEODATABASE
# ----------------------------
gdb = os.path.join(out_dir, "rasters.gdb")
if not arcpy.Exists(gdb):
    arcpy.management.CreateFileGDB(out_dir, "rasters.gdb")

out_gdb_ras = os.path.join(gdb, "Idomain_mask_30m")
safe_delete(out_gdb_ras)

arcpy.management.CopyRaster(
    in_raster=final,
    out_rasterdataset=out_gdb_ras,
    pixel_type="16_BIT_SIGNED",
    nodata_value="-32768"
)

print("✅ Saved to GDB:", out_gdb_ras)

# ----------------------------
# OPTIONAL: export to TIFF
# ----------------------------
out_tif = os.path.join(out_dir, "Idomain_mask_30m.tif")
safe_delete(out_tif)

try:
    arcpy.management.CopyRaster(
        in_raster=out_gdb_ras,
        out_rasterdataset=out_tif,
        pixel_type="16_BIT_SIGNED",
        nodata_value="-32768",
        format="TIFF"
    )
    print("✅ Exported TIFF:", out_tif)
except Exception as e:
    print("⚠️ TIFF export failed. Keep the GDB raster.")
    print("   Error:", e)

# ----------------------------
# SANITY CHECK
# ----------------------------
import numpy as np

ras = arcpy.Raster(out_gdb_ras)
print(f"\nRaster extent:  {ras.extent}")
print(f"Raster size:    {ras.height} rows × {ras.width} cols")
print(f"Cell size:      {ras.meanCellWidth:.1f} × {ras.meanCellHeight:.1f} m")

# Sample from center
sample_rows = min(1000, ras.height)
sample_cols = min(1000, ras.width)
r0 = (ras.height - sample_rows) // 2
c0 = (ras.width - sample_cols) // 2
ll = arcpy.Point(
    ras.extent.XMin + c0 * ras.meanCellWidth,
    ras.extent.YMax - (r0 + sample_rows) * ras.meanCellHeight
)
sample = arcpy.RasterToNumPyArray(
    out_gdb_ras, lower_left_corner=ll,
    ncols=sample_cols, nrows=sample_rows,
    nodata_to_value=-32768
)
vals, counts = np.unique(sample[sample != -32768], return_counts=True)
print(f"\n✅ Center sample ({sample_rows}×{sample_cols}):")
for v, c in zip(vals, counts):
    print(f"  {v:>4}  →  {c:>10,} cells")
print("Expected: [0, 1] — lakes INCLUDED in value=1")

DEM rows/cols: 45272 52180
DEM cellsize: 30.0 30.0
DEM SR: NAD_1983_Great_Lakes_Basin_Albers factoryCode: 3174
Output extent set to: D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\Ibound\extended_Bdry_final_GLB_Albers_buffered_20km.shp
✅ Saved to GDB: D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\Ibound\rasters.gdb\Idomain_mask_30m
✅ Exported TIFF: D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\Ibound\Idomain_mask_30m.tif

Raster extent:  278940 335790 1844340 1693980 NaN NaN NaN NaN
Raster size:    45273 rows × 52180 cols
Cell size:      30.0 × 30.0 m

✅ Center sample (1000×1000):
     1  →   1,000,000 cells
Expected: [0, 1] — lakes INCLUDED in value=1


# 4. Lake and shoreline Interaction

### 4.1 Constant head boundary
Identify shoreline-ring cells

Builds a one-cell-wide ring of land cells adjacent to lake cells. These shoreline-adjacent land cells are used as candidate constant-head cells.


In [ ]:
## Step A — Create a “shoreline ring” raster (land cells adjacent to lake)
import arcpy
import numpy as np
from arcpy.sa import Raster, Con, FocalStatistics, NbrRectangle

arcpy.CheckOutExtension("Spatial")
arcpy.env.overwriteOutput = True

mask = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\Costantheads\domain_water_mask_30m_buff20000m.tif"  # your output
out_ring = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\Costantheads\chd_shore_ring.tif"

r = Raster(mask)
lake = Con(r == -1, 1)             # 1 on lakes, NoData elsewhere
land = Con(r == 1, 1)              # 1 on land, NoData elsewhere

# Any land cell with a lake neighbor in a 3x3 window
lake_near = FocalStatistics(lake, NbrRectangle(3,3,"CELL"), "MAXIMUM", "DATA")
ring = Con((land == 1) & (lake_near == 1), 1, 0)

ring.save(out_ring)
print("✅ CHD shoreline ring raster:", out_ring)

### Convert shoreline-ring raster to points

Converts the shoreline-ring raster to point features so each candidate CHD cell has a cell-center point that can store head values.


In [ ]:
# Step B — Convert ring raster to points (CHD cell centers)

out_pts = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\Costantheads\CHD_cells_points.shp"

if arcpy.Exists(out_pts):
    arcpy.management.Delete(out_pts)

arcpy.conversion.RasterToPoint(out_ring, out_pts, "Value")
print("✅ CHD points:", out_pts)

### Sample DEM elevations at CHD points

Extracts DEM elevation to each CHD point so the assigned constant head can be checked against nearby land elevation.


In [ ]:
# Step C — Assign head values (better than constant 175)
# Simple + safe: head = min(lake_stage, DEM - 0.5)
import arcpy
from arcpy.sa import ExtractValuesToPoints

arcpy.CheckOutExtension("Spatial")
arcpy.env.overwriteOutput = True

out_pts = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\Costantheads\CHD_cells_points.shp"
dem     = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\DEM\DEM_merged_3174_30m.tif"

# IMPORTANT: output must be a NEW file (not the same as out_pts)
out_pts_dem = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\Costantheads\CHD_cells_points_dem.shp"

if arcpy.Exists(out_pts_dem):
    arcpy.management.Delete(out_pts_dem)

# optional but helpful
arcpy.management.RepairGeometry(out_pts)

# Extract DEM values to NEW points shapefile
ExtractValuesToPoints(out_pts, dem, out_pts_dem, interpolate_values="NONE")

print("✅ Wrote:", out_pts_dem)
print("Point count:", arcpy.management.GetCount(out_pts_dem)[0])
print("Fields:", [f.name for f in arcpy.ListFields(out_pts_dem)])

### Assign CHD head values

Creates or updates a `head` field at the CHD points. The assigned head is limited by both a lake stage and local DEM freeboard so heads do not exceed nearby land elevation.


In [ ]:
# Assign head values based on DEM + lake stage
import arcpy

arcpy.env.overwriteOutput = True

pts_dem = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\Costantheads\CHD_cells_points_dem.shp"

LAKE_STAGE = 177.0   # <-- choose your constant lake stage (m)
FREEBOARD  = 0.5     # head will be at least 0.5 m below DEM on land cells

# Add head field if missing
fields = [f.name for f in arcpy.ListFields(pts_dem)]
if "head" not in fields:
    arcpy.management.AddField(pts_dem, "head", "DOUBLE")

# Compute head from DEM sample (RASTERVALU)
with arcpy.da.UpdateCursor(pts_dem, ["RASTERVALU", "head"]) as cur:
    for dem_z, head in cur:
        if dem_z is None:
            new_head = LAKE_STAGE
        else:
            new_head = min(LAKE_STAGE, float(dem_z) - FREEBOARD)
        cur.updateRow((dem_z, new_head))

print("✅ head assigned to:", pts_dem)

# Quick sanity check (sample stats)
vals = []
with arcpy.da.SearchCursor(pts_dem, ["head"]) as cur:
    for (h,) in cur:
        if h is not None:
            vals.append(h)
        if len(vals) >= 200000:  # sample for speed
            break

print("Sample head min/max:", min(vals), max(vals))

### Rasterize CHD heads for visual QA/QC

Projects the CHD points if needed and rasterizes the `head` field back to the MODFLOW template grid. This raster is mainly for checking alignment and assigned values, not for direct MODFLOW input.


In [ ]:
# raterize the head points back to raster (for visual check, not required for Modflow) and make sure the crs adn resolution match the model grid (template raster)

import os
import arcpy

arcpy.env.overwriteOutput = True

# Inputs
pts_dem = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\Costantheads\CHD_cells_points_dem.shp"
template_raster = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\GRID_3174\template_1000m_epsg3174.tif"

out_dir = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\Costantheads"
os.makedirs(out_dir, exist_ok=True)

# Output raster (1000 m)
chd_head_ras = os.path.join(out_dir, "CHD_head_1000m.tif")

# --- CRS-safe: project points to template CRS if needed ---
sr_t = arcpy.Describe(template_raster).spatialReference
sr_p = arcpy.Describe(pts_dem).spatialReference

pts_use = pts_dem
if sr_p.factoryCode != sr_t.factoryCode:
    pts_proj = os.path.join(out_dir, "CHD_cells_points_dem_proj.shp")
    if arcpy.Exists(pts_proj):
        arcpy.management.Delete(pts_proj)
    arcpy.management.Project(pts_dem, pts_proj, sr_t)
    pts_use = pts_proj
    print("✅ Projected points to template CRS:", sr_t.factoryCode)

# Align output to model grid
arcpy.env.snapRaster = template_raster
arcpy.env.cellSize   = template_raster
arcpy.env.extent     = template_raster
arcpy.env.outputCoordinateSystem = sr_t

# Rasterize: MEAN head for each 1000m cell that has any points
if arcpy.Exists(chd_head_ras):
    arcpy.management.Delete(chd_head_ras)

arcpy.conversion.PointToRaster(
    in_features=pts_use,
    value_field="head",
    out_rasterdataset=chd_head_ras,
    cell_assignment="MEAN",
    priority_field="NONE",
    cellsize=arcpy.env.cellSize
)

arcpy.management.CalculateStatistics(chd_head_ras)
print("✅ CHD head raster:", chd_head_ras)

## 4.2 Create General head boundary flux
a Two way head dependent FLux to represent exchange of water between Great lakes and terrestrial
* this part should be run in climate python environment

In [ ]:
# ============================================================
# GREAT LAKES GHB PREP ONLY (NO gwf NEEDED YET)
# ============================================================
# This script:
# 1) Creates a 10 km inward buffer band inside each lake
# 2) Builds a temporary structured model grid (without gwf)
# 3) Intersects the lake band with the grid
# 4) Samples HK raster and computes first-pass conductance
# 5) Reads monthly lake stages from GLHYD_data_metric.csv
# 6) Writes prep outputs:
#       - lake band shapefile
#       - ghb cell shapefile
#       - ghb cell csv
#       - aligned monthly stage csv
#
# LATER:
# After you create gwf, you can read the prepared ghb cell table
# and build the MF6 GHB package from it.
# ============================================================

from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from shapely.geometry import Polygon
import flopy

from flopy.discretization import StructuredGrid
from flopy.utils.gridintersect import GridIntersect

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

# ============================================================
# USER INPUTS
# ============================================================

HEADS = r"S:\Data\Other_Data\Great_Lakes_Lake_Levels\Army_Core_Data\GLHYD_data_metric.csv"

GREAT_LAKES = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\Lakes\GreatLakes.shp"
GREAT_LAKES_10KM = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\Lakes\GreatLakes_buffer10km.shp"

HK_TIF = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\HK\HK_5band_mday_1000m.tif"

OUT_GHB_CELLS = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\GHBs\GreatLakes_GHB_cells.shp"
OUT_GHB_TABLE = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\GHBs\GreatLakes_GHB_cells.csv"
OUT_STAGE_TABLE = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\GHBs\GreatLakes_stage_monthly_for_model.csv"

LAKE_NAME_FIELD = "NAMEEN"
MODEL_CRS = "EPSG:3174"

# ============================================================
# GRID DEFINITION
# Fill these with your model grid information
# ============================================================

NLAY = 5
NROW = 250
NCOL = 305

# You can use either scalars or 1D arrays
DELR = 5000.0
DELC = 5000.0

# lower-left corner of the model grid
XORIGIN = 298957.5165494506
YORIGIN = 355789.11850404274

# grid rotation if any
ANGROT = 0.0

# ============================================================
# IDOMAIN
# If you already have idomain, assign it here:
#   IDOMAIN_3D = idomain
#
# If not, this script will assume all cells are active and put GHB
# in the top layer (k=0). You can rerun later once idomain is final.
# ============================================================

IDOMAIN_3D = None
# Example if you want all cells active now:
# IDOMAIN_3D = np.ones((NLAY, NROW, NCOL), dtype=int)

# ============================================================
# MODEL TIME RANGE
# ============================================================

MODEL_START = "2000-01-01"
MODEL_END   = "2025-12-01"
MODEL_MONTHS = pd.date_range(MODEL_START, MODEL_END, freq="MS")

# If model goes beyond observed stage record:
#   "extend_last" or "error"
STAGE_FUTURE_MODE = "extend_last"

# ============================================================
# LAKE BUFFER + CONDUCTANCE SETTINGS
# ============================================================

BUFFER_M = 10_000.0

BED_THICKNESS_M = 1.0
HK_BAND_FOR_K = 1
HK_TO_KV_DIVISOR = 10.0
MIN_COND = 1e-8
MIN_AREA_FRAC = 0.01

# ============================================================
# HELPERS
# ============================================================

def delete_shapefile(path_shp):
    path_shp = Path(path_shp)
    for ext in [".shp", ".shx", ".dbf", ".prj", ".cpg"]:
        p = path_shp.with_suffix(ext)
        if p.exists():
            try:
                p.unlink()
            except Exception:
                pass


def normalize_lake_name(name):
    s = str(name).strip().lower()
    s = s.replace("lake ", "")
    s = s.replace("_", " ")
    s = " ".join(s.split())

    mapping = {
        "superior": "Superior",
        "michigan": "Michigan",
        "huron": "Huron",
        "michigan-huron": "Michigan-Huron",
        "michigan huron": "Michigan-Huron",
        "erie": "Erie",
        "ontario": "Ontario",
        "st clair": "St. Clair",
        "st. clair": "St. Clair",
        "saint clair": "St. Clair",
    }
    return mapping.get(s, str(name).strip())


def get_valid_stage_lake_names():
    return ["Superior", "Michigan", "Huron", "Michigan-Huron", "Erie", "Ontario", "St. Clair"]


def get_stage_lookup_name(lake_name):
    lake_name = normalize_lake_name(lake_name)
    if lake_name in ["Michigan", "Huron", "Michigan-Huron"]:
        return "Michigan-Huron"
    elif lake_name == "Superior":
        return "Superior"
    elif lake_name == "Erie":
        return "Erie"
    elif lake_name == "Ontario":
        return "Ontario"
    elif lake_name == "St. Clair":
        return "St. Clair"
    return lake_name


def expand_spacing(value, n):
    """
    Convert scalar or iterable spacing to 1D numpy array of length n.
    """
    if np.isscalar(value):
        return np.full(n, float(value), dtype=float)
    arr = np.asarray(value, dtype=float).ravel()
    if len(arr) != n:
        raise ValueError(f"Spacing length {len(arr)} does not match expected {n}")
    return arr


def build_temp_modelgrid(nlay, nrow, ncol, delr, delc, xorigin, yorigin, angrot=0.0, crs=None):
    """
    Build a temporary FloPy StructuredGrid without gwf.
    """
    delr = expand_spacing(delr, ncol)
    delc = expand_spacing(delc, nrow)

    # top/botm not needed for intersection itself
    top = np.zeros((nrow, ncol), dtype=float)
    botm = np.zeros((nlay, nrow, ncol), dtype=float)

    mg = StructuredGrid(
        delc=delc,
        delr=delr,
        top=top,
        botm=botm,
        xoff=float(xorigin),
        yoff=float(yorigin),
        angrot=float(angrot),
        crs=crs,
    )
    return mg


def get_first_active_layer(idomain_3d, nlay, nrow, ncol):
    """
    Return first active layer k for each (i, j).
    If idomain_3d is None, assign k=0 everywhere.
    """
    if idomain_3d is None:
        return np.zeros((nrow, ncol), dtype=int)

    idomain_3d = np.asarray(idomain_3d)
    if idomain_3d.shape != (nlay, nrow, ncol):
        raise ValueError(
            f"IDOMAIN_3D shape {idomain_3d.shape} does not match "
            f"({nlay}, {nrow}, {ncol})"
        )

    firstk = np.full((nrow, ncol), -1, dtype=int)
    for k in range(nlay):
        mask = (firstk < 0) & (idomain_3d[k] > 0)
        firstk[mask] = k
    return firstk


def sample_raster_at_xy(raster_path, xy, band=1):
    vals = []
    with rasterio.open(raster_path) as src:
        for v in src.sample(xy, indexes=band):
            if v is None or len(v) == 0:
                vals.append(np.nan)
            else:
                vals.append(float(v[0]))
    return np.array(vals, dtype=float)


def make_inward_lake_band(
    in_lakes,
    out_band,
    buffer_m=10000.0,
    target_crs="EPSG:3174",
    lake_name_field="NAMEEN",
):
    """
    Create a 10 km band INSIDE each lake.
    """
    lakes = gpd.read_file(in_lakes)

    if lakes.crs is None:
        raise ValueError("Great_lakes shapefile has no CRS.")

    lakes = lakes.to_crs(target_crs)

    if lake_name_field not in lakes.columns:
        raise ValueError(
            f"Lake name field '{lake_name_field}' not found.\n"
            f"Available columns: {list(lakes.columns)}"
        )

    lakes = lakes[[lake_name_field, "geometry"]].copy()
    lakes["lake_name"] = lakes[lake_name_field].apply(normalize_lake_name)

    print(f"Using lake name field: {lake_name_field}")
    print("\nUnique names found in shapefile:")
    print(sorted(lakes["lake_name"].dropna().unique()))

    valid_lakes = get_valid_stage_lake_names()
    lakes = lakes[lakes["lake_name"].isin(valid_lakes)].copy()

    if len(lakes) == 0:
        raise ValueError("No valid lakes remained after filtering.")

    lakes = lakes[["lake_name", "geometry"]].dissolve(by="lake_name", as_index=False)

    out_rows = []
    for _, row in lakes.iterrows():
        lake_name = row["lake_name"]
        geom = row.geometry

        inner = geom.buffer(-buffer_m)
        if inner.is_empty:
            print(f"Skipping {lake_name}: inward buffer removed polygon.")
            continue

        band = geom.difference(inner)
        if band.is_empty:
            print(f"Skipping {lake_name}: resulting band is empty.")
            continue

        out_rows.append({"lake_name": lake_name, "geometry": band})

    if len(out_rows) == 0:
        raise ValueError("No inward lake bands were created.")

    band_gdf = gpd.GeoDataFrame(out_rows, crs=target_crs)

    out_band = Path(out_band)
    out_band.parent.mkdir(parents=True, exist_ok=True)
    delete_shapefile(out_band)
    band_gdf.to_file(out_band)

    print(f"\nSaved 10 km inward lake band:\n  {out_band}")
    return band_gdf

def read_glhyd_monthly_csv(csv_path):
    """
    Read Great Lakes monthly stage CSV robustly.

    Handles:
    - comment lines starting with '#'
    - blank lines before header
    - BOM / extra spaces in column names
    - header row not necessarily first line
    """

    from pathlib import Path

    csv_path = Path(csv_path)

    # --------------------------------------------------------
    # 1) detect header row
    # --------------------------------------------------------
    with open(csv_path, "r", encoding="utf-8-sig", errors="ignore") as f:
        lines = f.readlines()

    header_idx = None
    for i, line in enumerate(lines[:50]):
        line_clean = line.strip().lower()

        if not line_clean:
            continue
        if line_clean.startswith("#"):
            continue

        if ("year" in line_clean) and (
            "superior" in line_clean
            or "michigan" in line_clean
            or "erie" in line_clean
            or "ontario" in line_clean
        ):
            header_idx = i
            break

    if header_idx is None:
        raise ValueError(
            "Could not detect the header row in the stage CSV."
        )

    # --------------------------------------------------------
    # 2) read CSV from detected header row
    # --------------------------------------------------------
    df = pd.read_csv(
        csv_path,
        skiprows=header_idx,
        comment="#",
        encoding="utf-8-sig"
    )

    # --------------------------------------------------------
    # 3) clean column names
    # --------------------------------------------------------
    df.columns = [str(c).replace("\ufeff", "").strip() for c in df.columns]

    norm_cols = {}
    for c in df.columns:
        key = str(c).strip().lower().replace("_", " ")
        key = " ".join(key.split())
        norm_cols[key] = c

    # --------------------------------------------------------
    # 4) find needed columns
    # --------------------------------------------------------
    month_col = norm_cols.get("month", None)
    year_col = norm_cols.get("year", None)

    if month_col is None or year_col is None:
        raise ValueError(
            f"Could not find month/year columns. Detected columns: {list(df.columns)}"
        )

    def find_stage_col(possible_names):
        for p in possible_names:
            p_norm = " ".join(p.strip().lower().replace("_", " ").split())
            if p_norm in norm_cols:
                return norm_cols[p_norm]
        return None

    superior_col = find_stage_col(["Superior"])
    michhur_col  = find_stage_col(["Michigan-Huron", "Michigan Huron"])
    stclair_col  = find_stage_col(["St. Clair", "St Clair", "Saint Clair"])
    erie_col     = find_stage_col(["Erie"])
    ontario_col  = find_stage_col(["Ontario"])

    needed = {
        "Superior": superior_col,
        "Michigan-Huron": michhur_col,
        "Erie": erie_col,
        "Ontario": ontario_col,
    }
    missing = [k for k, v in needed.items() if v is None]
    if missing:
        raise ValueError(
            f"Missing required lake-stage columns: {missing}\n"
            f"Detected columns: {list(df.columns)}"
        )

    # --------------------------------------------------------
    # 5) parse month/year
    # --------------------------------------------------------
    month_map = {
        "jan": 1, "feb": 2, "mar": 3, "apr": 4,
        "may": 5, "jun": 6, "jul": 7, "aug": 8,
        "sep": 9, "sept": 9, "oct": 10, "nov": 11, "dec": 12,
    }

    month_raw = df[month_col].astype(str).str.strip().str.lower()
    year_raw = pd.to_numeric(df[year_col], errors="coerce")

    month_num = pd.to_numeric(month_raw, errors="coerce")
    month_num = month_num.where(month_num.notna(), month_raw.map(month_map))

    if month_num.isna().any():
        bad_months = df.loc[month_num.isna(), month_col].unique()
        raise ValueError(f"Unrecognized month values in CSV: {bad_months}")

    if year_raw.isna().any():
        bad_years = df.loc[year_raw.isna(), year_col].unique()
        raise ValueError(f"Unrecognized year values in CSV: {bad_years}")

    df["date"] = pd.to_datetime(
        {
            "year": year_raw.astype(int),
            "month": month_num.astype(int),
            "day": 1
        }
    )

    # --------------------------------------------------------
    # 6) build stage table CORRECTLY
    # --------------------------------------------------------
    keep_cols = {
        superior_col: "Superior",
        michhur_col: "Michigan-Huron",
        erie_col: "Erie",
        ontario_col: "Ontario",
    }
    if stclair_col is not None:
        keep_cols[stclair_col] = "St. Clair"

    out = df[["date"] + list(keep_cols.keys())].copy()
    out = out.rename(columns=keep_cols)

    for c in keep_cols.values():
        out[c] = pd.to_numeric(out[c], errors="coerce")

    out = out.set_index("date").sort_index()
    out = out.groupby(out.index).mean()

    print("\nLoaded stage table successfully.")
    print("Detected columns:", list(df.columns))
    print("Stage range:", out.index.min(), "to", out.index.max())
    print(out.head())

    return out


def align_stages_to_model_months(monthly_stages, model_months, future_mode="extend_last"):
    """
    Align lake stage table to model month-start dates.
    """
    # force model months to month-start timestamps
    model_months = pd.DatetimeIndex(model_months)
    model_months = model_months.to_period("M").to_timestamp(how="start")

    aligned = monthly_stages.copy()
    aligned.index = pd.DatetimeIndex(aligned.index).to_period("M").to_timestamp(how="start")

    aligned = aligned.reindex(model_months)
    aligned = aligned.interpolate(limit_direction="both")

    if aligned.isna().any().any():
        if future_mode == "error":
            raise ValueError(
                "Lake stage table does not fully cover the model period.\n"
                f"Stage range: {monthly_stages.index.min()} to {monthly_stages.index.max()}\n"
                f"Model range: {model_months.min()} to {model_months.max()}"
            )
        elif future_mode == "extend_last":
            aligned = aligned.ffill().bfill()
        else:
            raise ValueError(f"Unknown future_mode: {future_mode}")

    return aligned

def build_grid_cell_polygons(modelgrid, firstk):
    """
    Convert the structured model grid to a GeoDataFrame of active cell polygons.
    """
    if modelgrid.grid_type != "structured":
        raise ValueError("This function assumes a structured grid.")

    nrow = modelgrid.nrow
    ncol = modelgrid.ncol
    delr = np.asarray(modelgrid.delr)
    delc = np.asarray(modelgrid.delc)

    rows = []
    for i in range(nrow):
        for j in range(ncol):
            k = int(firstk[i, j])
            if k < 0:
                continue

            verts = modelgrid.get_cell_vertices(i, j)
            poly = Polygon(verts)

            if not poly.is_valid:
                poly = poly.buffer(0)
            if poly.is_empty:
                continue

            rows.append(
                {
                    "k": k,
                    "i": i,
                    "j": j,
                    "cell_area_m2": float(delr[j] * delc[i]),
                    "geometry": poly,
                }
            )

    if len(rows) == 0:
        raise ValueError("No active grid cells were converted to polygons.")

    grid_gdf = gpd.GeoDataFrame(rows, geometry="geometry", crs=modelgrid.crs)
    return grid_gdf


def build_ghb_cell_table_from_grid(
    modelgrid,
    firstk,
    lake_band_gdf,
    hk_tif,
    bed_thickness_m=1.0,
    hk_band=1,
    hk_to_kv_divisor=10.0,
    min_cond=1e-8,
    min_area_frac=0.01,
    out_cells_path=None,
    out_table_path=None,
):
    """
    Intersect lake buffer band with structured grid polygons and compute conductance.
    This version does NOT use GridIntersect, so it is more version-safe.
    """
    if modelgrid.grid_type != "structured":
        raise ValueError("This script assumes a structured grid.")

    # build active cell polygons once
    grid_cells = build_grid_cell_polygons(modelgrid, firstk)

    xcc = np.asarray(modelgrid.xcellcenters)
    ycc = np.asarray(modelgrid.ycellcenters)

    out = []

    for _, row in lake_band_gdf.iterrows():
        lake_name = normalize_lake_name(row["lake_name"])
        geom = row.geometry

        print(f"\nIntersecting lake band for: {lake_name}")

        lake_one = gpd.GeoDataFrame(
            [{"lake_name_src": lake_name, "geometry": geom}],
            geometry="geometry",
            crs=lake_band_gdf.crs,
        )

        inter = gpd.overlay(grid_cells, lake_one, how="intersection", keep_geom_type=True)

        if inter is None or len(inter) == 0:
            print(f"  No intersected cells for {lake_name}")
            continue

        inter["overlap_m2"] = inter.geometry.area

        # remove tiny slivers
        inter = inter.loc[inter["overlap_m2"] >= (inter["cell_area_m2"] * min_area_frac)].copy()
        if len(inter) == 0:
            print(f"  All intersections were below min area fraction for {lake_name}")
            continue

        # sample HK at original cell centers
        xy = [(float(xcc[i, j]), float(ycc[i, j])) for i, j in zip(inter["i"], inter["j"])]
        hk_vals = sample_raster_at_xy(hk_tif, xy, band=hk_band)

        hk_vals = np.where(np.isfinite(hk_vals) & (hk_vals > 0), hk_vals, np.nan)
        hk_med = np.nanmedian(hk_vals)
        if not np.isfinite(hk_med):
            raise ValueError(f"No valid HK values could be sampled from {hk_tif}")

        hk_vals = np.where(np.isfinite(hk_vals), hk_vals, hk_med)
        kv_vals = hk_vals / float(hk_to_kv_divisor)

        # conductance = Kv * A / T
        cond_vals = kv_vals * inter["overlap_m2"].to_numpy() / float(bed_thickness_m)
        cond_vals = np.where(cond_vals > min_cond, cond_vals, min_cond)

        tmp = gpd.GeoDataFrame(
            {
                "lake_name": lake_name,
                "stage_name": [get_stage_lookup_name(lake_name)] * len(inter),
                "k": inter["k"].astype(int),
                "i": inter["i"].astype(int),
                "j": inter["j"].astype(int),
                "overlap_m2": inter["overlap_m2"].astype(float),
                "cell_area_m2": inter["cell_area_m2"].astype(float),
                "hk_mday": hk_vals.astype(float),
                "kv_mday": kv_vals.astype(float),
                "cond": cond_vals.astype(float),
                "geometry": inter.geometry,
            },
            geometry="geometry",
            crs=lake_band_gdf.crs,
        )

        out.append(tmp)

        print(
            f"  {lake_name}: {len(tmp):,} cells | "
            f"cond min/max = {tmp['cond'].min():.3e} / {tmp['cond'].max():.3e}"
        )

    if len(out) == 0:
        raise ValueError("No GHB cells were created.")

    ghb_cells = pd.concat(out, ignore_index=True)
    ghb_cells = gpd.GeoDataFrame(ghb_cells, geometry="geometry", crs=lake_band_gdf.crs)

    ghb_cells["bname"] = [
        f"{stage_nm[:8]}_{int(i)}_{int(j)}"
        for stage_nm, i, j in zip(ghb_cells["stage_name"], ghb_cells["i"], ghb_cells["j"])
    ]

    if out_cells_path is not None:
        delete_shapefile(out_cells_path)
        ghb_cells.to_file(out_cells_path)
        print(f"\nSaved GHB cell shapefile:\n  {out_cells_path}")

    if out_table_path is not None:
        pd.DataFrame(ghb_cells.drop(columns="geometry")).to_csv(out_table_path, index=False)
        print(f"Saved GHB cell table:\n  {out_table_path}")

    return ghb_cells


# ============================================================
# MAIN PREP
# ============================================================

# 1) build temporary model grid
mg = build_temp_modelgrid(
    nlay=NLAY,
    nrow=NROW,
    ncol=NCOL,
    delr=DELR,
    delc=DELC,
    xorigin=XORIGIN,
    yorigin=YORIGIN,
    angrot=ANGROT,
    crs=MODEL_CRS,
)

# 2) first active layer
firstk = get_first_active_layer(
    idomain_3d=IDOMAIN_3D,
    nlay=NLAY,
    nrow=NROW,
    ncol=NCOL,
)

if IDOMAIN_3D is None:
    print("\nIDOMAIN_3D is None -> assigning all GHB cells to k=0 for now.")

# 3) create 10 km inward lake band
lake_band = make_inward_lake_band(
    in_lakes=GREAT_LAKES,
    out_band=GREAT_LAKES_10KM,
    buffer_m=BUFFER_M,
    target_crs=MODEL_CRS,
    lake_name_field=LAKE_NAME_FIELD,
)

# 4) intersect with grid and compute conductance
ghb_cells = build_ghb_cell_table_from_grid(
    modelgrid=mg,
    firstk=firstk,
    lake_band_gdf=lake_band,
    hk_tif=HK_TIF,
    bed_thickness_m=BED_THICKNESS_M,
    hk_band=HK_BAND_FOR_K,
    hk_to_kv_divisor=HK_TO_KV_DIVISOR,
    min_cond=MIN_COND,
    min_area_frac=MIN_AREA_FRAC,
    out_cells_path=OUT_GHB_CELLS,
    out_table_path=OUT_GHB_TABLE,
)

print("\nGHB summary by lake:")
print(
    ghb_cells.groupby("lake_name")
    .agg(
        n_cells=("lake_name", "size"),
        area_km2=("overlap_m2", lambda s: s.sum() / 1e6),
        cond_min=("cond", "min"),
        cond_max=("cond", "max"),
    )
)

# 5) read and align monthly stages
monthly_stages = read_glhyd_monthly_csv(HEADS)
monthly_stages_model = align_stages_to_model_months(
    monthly_stages=monthly_stages,
    model_months=MODEL_MONTHS,
    future_mode=STAGE_FUTURE_MODE,
)

Path(OUT_STAGE_TABLE).parent.mkdir(parents=True, exist_ok=True)
monthly_stages_model.to_csv(OUT_STAGE_TABLE, index=True)
print(f"\nSaved aligned monthly stage table:\n  {OUT_STAGE_TABLE}")

print("\nStage columns available:")
print(list(monthly_stages_model.columns))

print("\n✅ GHB prep finished.")
print("Outputs created:")
print(f"  Lake band:   {GREAT_LAKES_10KM}")
print(f"  GHB cells:   {OUT_GHB_CELLS}")
print(f"  GHB table:   {OUT_GHB_TABLE}")
print(f"  Stage table: {OUT_STAGE_TABLE}")

## 5- Create aligned bottom-elevation rasters

Converts the bedrock-surface ASCII grid to raster, aligns it to the MODFLOW template grid, and builds bottom-elevation rasters for deeper model layers.


In [1]:
import os
import numpy as np
import arcpy
from arcpy.sa import Raster, ExtractByMask, SetNull

arcpy.CheckOutExtension("Spatial")
arcpy.env.overwriteOutput = True

# --- inputs ---
asc_bedrock     = r"S:\Data\Other_Data\Xu_2021\Data\01_Bedrock_Topography\integrated_bedrock_surface_elevation.asc"
template_raster = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\DEM\DEM_extended20kmbdr_1000m.tif"
boundary_shp    = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\Ibound\extended_Bdry_final_GLB_Albers_exported.shp"

# --- thicknesses ---
FRAC_THICK_M    = 50.0
BEDROCK_THICK_M = 500.0
TOTAL_THICK_M   = FRAC_THICK_M + BEDROCK_THICK_M

out_dir = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\Bottom"
os.makedirs(out_dir, exist_ok=True)

bedrock_raw  = os.path.join(out_dir, "bedrock_surface_raw.tif")
bedrock_aln  = os.path.join(out_dir, "bedrock_surface_aligned.tif")
bedrock_clp  = os.path.join(out_dir, "bedrock_surface_clipped.tif")
boundary_prj = os.path.join(out_dir, "boundary_3174.shp")
modelbottom  = os.path.join(out_dir, "modelbottom.tif")

# ---- CRS definitions ----
sr_3174  = arcpy.SpatialReference(3174)
sr_input = arcpy.SpatialReference(26917)  # NAD83 UTM Zone 17N

# ---- Environment ----
arcpy.env.snapRaster             = template_raster
arcpy.env.cellSize               = template_raster
arcpy.env.extent                 = template_raster
arcpy.env.outputCoordinateSystem = sr_3174

# ---- Helper: raster info ----
def raster_info(path, label):
    arr = arcpy.RasterToNumPyArray(path)
    desc = arcpy.Describe(path)
    nd = desc.noDataValue
    print(f"\n  [{label}]")
    print(f"  CRS        : {desc.spatialReference.name} (EPSG:{desc.spatialReference.factoryCode})")
    print(f"  Shape      : {arr.shape}")
    print(f"  Dtype      : {arr.dtype}")
    print(f"  NoData val : {nd}")
    if nd is not None:
        valid = arr[arr != nd]
    else:
        mask = np.ones(arr.shape, dtype=bool)
        for test in [-9999, -9999.0, 3.4028235e+38, -3.4028235e+38]:
            mask &= (arr != test)
        valid = arr[mask]
    print(f"  Valid cells: {valid.size:,} of {arr.size:,}")
    if valid.size > 0:
        print(f"  Valid min  : {valid.min():.2f} m")
        print(f"  Valid max  : {valid.max():.2f} m")
    else:
        print(f"  ⚠️  No valid cells found!")
    return valid.size > 0

# ============================================================
# STEP 0 — Reproject boundary to EPSG:3174 if needed
# ============================================================
print("=" * 55)
print("STEP 0: Checking boundary CRS...")
print("=" * 55)

sr_bdry = arcpy.Describe(boundary_shp).spatialReference
print(f"  Boundary CRS: {sr_bdry.name} (EPSG:{sr_bdry.factoryCode})")

if sr_bdry.factoryCode != 3174:
    print(f"  Reprojecting boundary to EPSG:3174...")
    if arcpy.Exists(boundary_prj):
        arcpy.management.Delete(boundary_prj)
    arcpy.management.Project(boundary_shp, boundary_prj, sr_3174)
    boundary_use = boundary_prj
else:
    boundary_use = boundary_shp

ext_bdry = arcpy.Describe(boundary_use).extent
print(f"  Boundary extent (3174):")
print(f"    XMin={ext_bdry.XMin:.0f}  YMin={ext_bdry.YMin:.0f}")
print(f"    XMax={ext_bdry.XMax:.0f}  YMax={ext_bdry.YMax:.0f}")
print(f"  ✅ Boundary ready.")

# ============================================================
# STEP 1 — ASCII to Raster + Define Projection
# ============================================================
print("\n" + "=" * 55)
print("STEP 1: Converting ASC to raster...")
print("=" * 55)

for p in [bedrock_raw, bedrock_aln, bedrock_clp, modelbottom]:
    if arcpy.Exists(p):
        arcpy.management.Delete(p)

# Convert ASC — environment CRS is intentionally NOT set here
# so the raw raster keeps its native coordinates
arcpy.env.outputCoordinateSystem = None
arcpy.conversion.ASCIIToRaster(asc_bedrock, bedrock_raw, "FLOAT")

# ---- KEY FIX: Define the correct CRS on the raw raster ----
# The ASC has no embedded CRS (EPSG:0) but coordinates confirm UTM 17N
print("  Defining projection on raw raster (EPSG:26917)...")
arcpy.management.DefineProjection(bedrock_raw, sr_input)

arcpy.management.CalculateStatistics(bedrock_raw, skip_existing="OVERWRITE")
arcpy.management.BuildPyramids(bedrock_raw, skip_existing="OVERWRITE")

# Verify CRS and extent after defining
sr_raw = arcpy.Describe(bedrock_raw).spatialReference
ext_raw = arcpy.Describe(bedrock_raw).extent
print(f"  CRS after define: {sr_raw.name} (EPSG:{sr_raw.factoryCode})")
print(f"  Extent: XMin={ext_raw.XMin:.0f}  YMin={ext_raw.YMin:.0f}")
print(f"          XMax={ext_raw.XMax:.0f}  YMax={ext_raw.YMax:.0f}")

has_valid = raster_info(bedrock_raw, "bedrock_surface_raw")
if not has_valid:
    raise RuntimeError("❌ bedrock_raw has no valid cells — check ASC file.")
print("  ✅ Raw bedrock raster created with correct CRS.")

# Restore environment CRS for remaining steps
arcpy.env.outputCoordinateSystem = sr_3174

# ============================================================
# STEP 2 — Project to EPSG:3174
# ============================================================
print("\n" + "=" * 55)
print("STEP 2: Projecting bedrock to EPSG:3174...")
print("=" * 55)

arcpy.management.ProjectRaster(
    in_raster=bedrock_raw,
    out_raster=bedrock_aln,
    out_coor_system=sr_3174,
    resampling_type="BILINEAR",
    cell_size=arcpy.Describe(template_raster).meanCellWidth,
    in_coor_system=sr_input
)
arcpy.management.CalculateStatistics(bedrock_aln, skip_existing="OVERWRITE")
arcpy.management.BuildPyramids(bedrock_aln, skip_existing="OVERWRITE")

ext_aln = arcpy.Describe(bedrock_aln).extent
print(f"  Bedrock aligned extent (3174):")
print(f"    XMin={ext_aln.XMin:.0f}  YMin={ext_aln.YMin:.0f}")
print(f"    XMax={ext_aln.XMax:.0f}  YMax={ext_aln.YMax:.0f}")

has_valid = raster_info(bedrock_aln, "bedrock_surface_aligned")
if not has_valid:
    raise RuntimeError("❌ bedrock_aln has no valid cells.")
print("  ✅ Bedrock projected to EPSG:3174.")

# ============================================================
# STEP 3 — Verify overlap then clip
# ============================================================
print("\n" + "=" * 55)
print("STEP 3: Verifying overlap and clipping...")
print("=" * 55)

ext_bed  = arcpy.Describe(bedrock_aln).extent
ext_buse = arcpy.Describe(boundary_use).extent

print(f"  Bedrock extent : XMin={ext_bed.XMin:.0f}  YMin={ext_bed.YMin:.0f}")
print(f"                   XMax={ext_bed.XMax:.0f}  YMax={ext_bed.YMax:.0f}")
print(f"  Boundary extent: XMin={ext_buse.XMin:.0f}  YMin={ext_buse.YMin:.0f}")
print(f"                   XMax={ext_buse.XMax:.0f}  YMax={ext_buse.YMax:.0f}")

x_overlap = ext_bed.XMin < ext_buse.XMax and ext_bed.XMax > ext_buse.XMin
y_overlap = ext_bed.YMin < ext_buse.YMax and ext_bed.YMax > ext_buse.YMin
print(f"  X overlap: {x_overlap}")
print(f"  Y overlap: {y_overlap}")

if not (x_overlap and y_overlap):
    raise RuntimeError("❌ Still no overlap — check source data extents above.")
print("  ✅ Extents overlap confirmed.")

# Clean extreme nodata values before clipping
bed_ras   = Raster(bedrock_aln)
bed_clean = SetNull(bed_ras > 9000, SetNull(bed_ras < -5000, bed_ras))

# Clip to boundary
bed_clip = ExtractByMask(bed_clean, boundary_use)
bed_clip.save(bedrock_clp)
arcpy.management.CalculateStatistics(bedrock_clp, skip_existing="OVERWRITE")
arcpy.management.BuildPyramids(bedrock_clp, skip_existing="OVERWRITE")

has_valid = raster_info(bedrock_clp, "bedrock_surface_clipped")
if not has_valid:
    raise RuntimeError("❌ bedrock_clp has no valid cells after clip.")
print("  ✅ Bedrock clipped to boundary.")

# ============================================================
# STEP 4 — Calculate model bottom
# ============================================================
print("\n" + "=" * 55)
print("STEP 4: Calculating model bottom...")
print("=" * 55)

result = Raster(bedrock_clp) - TOTAL_THICK_M
result.save(modelbottom)
arcpy.management.CalculateStatistics(modelbottom, skip_existing="OVERWRITE")
arcpy.management.BuildPyramids(modelbottom, skip_existing="OVERWRITE")

has_valid = raster_info(modelbottom, "modelbottom")
if not has_valid:
    raise RuntimeError("❌ modelbottom has no valid cells.")

# ============================================================
# STEP 5 — Final summary
# ============================================================
print("\n" + "=" * 55)
print("✅ ALL STEPS COMPLETE")
print("=" * 55)
sr_final = arcpy.Describe(modelbottom).spatialReference
print(f"  Output      : {modelbottom}")
print(f"  CRS         : {sr_final.name} (EPSG:{sr_final.factoryCode})")
print(f"  Frac thick  : {FRAC_THICK_M} m")
print(f"  Bed thick   : {BEDROCK_THICK_M} m")
print(f"  Total thick : {TOTAL_THICK_M} m")

# ---- Reset environment ----
arcpy.env.snapRaster             = None
arcpy.env.cellSize               = None
arcpy.env.extent                 = None
arcpy.env.outputCoordinateSystem = None


STEP 0: Checking boundary CRS...
  Boundary CRS: NAD_1983_Great_Lakes_Basin_Albers (EPSG:3174)
  Boundary extent (3174):
    XMin=298958  YMin=355789
    XMax=1824341  YMax=1673951
  ✅ Boundary ready.

STEP 1: Converting ASC to raster...
  Defining projection on raw raster (EPSG:26917)...
  CRS after define: NAD_1983_UTM_Zone_17N (EPSG:26917)
  Extent: XMin=-540025  YMin=4467807
          XMax=1114967  YMax=5697773

  [bedrock_surface_raw]
  CRS        : NAD_1983_UTM_Zone_17N (EPSG:26917)
  Shape      : (14750, 19847)
  Dtype      : float32
  NoData val : -3.4028234663852886e+38
  Valid cells: 242,232,506 of 292,743,250
  Valid min  : -622.22 m
  Valid max  : 1479.00 m
  ✅ Raw bedrock raster created with correct CRS.

STEP 2: Projecting bedrock to EPSG:3174...
  Bedrock aligned extent (3174):
    XMin=223940  YMin=389940
    XMax=1897940  YMax=1680940

  [bedrock_surface_aligned]
  CRS        : NAD_1983_Great_Lakes_Basin_Albers (EPSG:3174)
  Shape      : (1291, 1674)
  Dtype      : flo

In [6]:
# Quick diagnostic on raw raster extent
bedrock_raw = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\Bottom\bedrock_surface_raw.tif"

ext = arcpy.Describe(bedrock_raw).extent
print(f"Raw bedrock extent:")
print(f"  XMin={ext.XMin:.4f}  YMin={ext.YMin:.4f}")
print(f"  XMax={ext.XMax:.4f}  YMax={ext.YMax:.4f}")
print(f"  Width ={ext.XMax - ext.XMin:.4f}")
print(f"  Height={ext.YMax - ext.YMin:.4f}")

Raw bedrock extent:
  XMin=-540024.6160  YMin=4467807.4438
  XMax=1114966.6520  YMax=5697772.7385
  Width =1654991.2680
  Height=1229965.2947


In [7]:
# Once you know the correct EPSG, replace 26917 with the right one
# Example if geographic NAD83:
sr_input = arcpy.SpatialReference(4269)

# OR if already 3174:
sr_input = arcpy.SpatialReference(3174)

# Then define it on the raw raster before projecting
arcpy.management.DefineProjection(bedrock_raw, sr_input)

# Verify
sr_check = arcpy.Describe(bedrock_raw).spatialReference
print(f"Defined CRS: {sr_check.name} (EPSG:{sr_check.factoryCode})")
ext = arcpy.Describe(bedrock_raw).extent
print(f"XMin={ext.XMin:.2f}  YMin={ext.YMin:.2f}")
print(f"XMax={ext.XMax:.2f}  YMax={ext.YMax:.2f}")

Defined CRS: NAD_1983_Great_Lakes_Basin_Albers (EPSG:3174)
XMin=-540024.62  YMin=4467807.44
XMax=1114966.65  YMax=5697772.74


In [4]:
import arcpy
import os

bedrock_aln  = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\Bottom\bedrock_surface_aligned.tif"
boundary_shp = r"S:\Projects\Active\GLB_LHM\LHM_inputs\Boundary\extendedBdry_jan26_adk.shp"
bedrock_raw  = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\Bottom\bedrock_surface_raw.tif"

# ---- Check all CRS ----
sr_bed_raw = arcpy.Describe(bedrock_raw).spatialReference
sr_bed_aln = arcpy.Describe(bedrock_aln).spatialReference
sr_bdry    = arcpy.Describe(boundary_shp).spatialReference

print("CRS CHECK:")
print(f"  bedrock_raw : {sr_bed_raw.name} (EPSG:{sr_bed_raw.factoryCode})")
print(f"  bedrock_aln : {sr_bed_aln.name} (EPSG:{sr_bed_aln.factoryCode})")
print(f"  boundary    : {sr_bdry.name} (EPSG:{sr_bdry.factoryCode})")

# ---- Print all extents ----
ext_raw  = arcpy.Describe(bedrock_raw).extent
ext_aln  = arcpy.Describe(bedrock_aln).extent
ext_bdry = arcpy.Describe(boundary_shp).extent

print("\nEXTENT CHECK (raw bedrock — original CRS):")
print(f"  XMin={ext_raw.XMin:.2f}  YMin={ext_raw.YMin:.2f}")
print(f"  XMax={ext_raw.XMax:.2f}  YMax={ext_raw.YMax:.2f}")

print("\nEXTENT CHECK (bedrock_aln — after ProjectRaster):")
print(f"  XMin={ext_aln.XMin:.2f}  YMin={ext_aln.YMin:.2f}")
print(f"  XMax={ext_aln.XMax:.2f}  YMax={ext_aln.YMax:.2f}")

print("\nEXTENT CHECK (boundary shapefile):")
print(f"  XMin={ext_bdry.XMin:.2f}  YMin={ext_bdry.YMin:.2f}")
print(f"  XMax={ext_bdry.XMax:.2f}  YMax={ext_bdry.YMax:.2f}")

CRS CHECK:
  bedrock_raw : NAD_1983_UTM_Zone_17N (EPSG:26917)
  bedrock_aln : NAD_1983_Great_Lakes_Basin_Albers (EPSG:3174)
  boundary    : NAD_1983_UTM_Zone_17N (EPSG:26917)

EXTENT CHECK (raw bedrock — original CRS):
  XMin=-540024.62  YMin=4467807.44
  XMax=1114966.65  YMax=5697772.74

EXTENT CHECK (bedrock_aln — after ProjectRaster):
  XMin=-540060.00  YMin=4466940.00
  XMax=1115940.00  YMax=5697940.00

EXTENT CHECK (boundary shapefile):
  XMin=-467654.72  YMin=4406978.19
  XMax=1050167.22  YMax=5670086.89


## 5. Active-domain raster
Create a raster representation of the model domain boundary.


### Rasterize the active-domain boundary

Converts the polygon model boundary to a raster aligned to the DEM grid. This is useful for creating the active model domain / ibound-style mask.


In [ ]:
Ibound = r'S:\Projects\Active\GLB_LHM\LHM_inputs\Boundary\extendedBdry_jan26_adk.shp'
Ibound_ras = r'D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\Ibound\extendedBdry_jan26_adk.tif'
dem = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\DEM\DEM_merged_3174_30m.tif"

import arcpy
arcpy.env.overwriteOutput = True
arcpy.env.snapRaster = dem
arcpy.env.cellSize = dem

arcpy.conversion.PolygonToRaster(
    in_features=Ibound,
    value_field="FID",
    out_rasterdataset=Ibound_ras,
    cell_assignment="MAXIMUM_AREA",
    priority_field="NONE",
    cellsize=dem
)
print("✅ Rasterized Ibound to:", Ibound_ras)

# 6. Recharge processing
First create and verify the blended recharge products, then convert the annual recharge product to MODFLOW-ready rasters.


### Create monthly NOAH–VIC blended NetCDF files

Builds monthly blended recharge NetCDF files by year from NOAH and VIC data. This is the upstream forcing-preparation step before converting recharge to model rasters.


In [ ]:
from pathlib import Path
import re
import xarray as xr
import numpy as np

def write_monthly_blend_nc_by_year(
    noah_root: Path,
    vic_root: Path,
    out_root: Path,
    years=range(2000, 2026),
    var_name="Qsb",
    scale_factor=0.75,
    out_var_name=None,          # default keeps same var name (Qsb) for easy downstream use
    file_glob="*.nc*",
    filename_prefix="BLEND",    # output file prefix
):
    """
    Create monthly blended NetCDFs in year folders:

      out_root/2000/BLEND_Qsb_A200001.nc
      out_root/2000/BLEND_Qsb_A200002.nc
      ...
      out_root/2025/BLEND_Qsb_A202512.nc

    Blend per month:
        blend = 0.7 * (NOAH + VIC)/2

    Matching is done by parsing AYYYYMM from file names (e.g., ...A201908...nc).
    """

    noah_root = Path(noah_root)
    vic_root  = Path(vic_root)
    out_root  = Path(out_root)
    out_root.mkdir(parents=True, exist_ok=True)

    if out_var_name is None:
        out_var_name = var_name  # keep same variable name unless you want Qsb_blend

    # Parse AYYYYMM anywhere in filename
    yyyymm_re = re.compile(r"A(\d{6})")

    def index_files_by_yyyymm(year_dir: Path):
        """Return {yyyymm: Path} for all files in the year directory."""
        out = {}
        for f in sorted(year_dir.glob(file_glob)):
            m = yyyymm_re.search(f.name)
            if not m:
                continue
            yyyymm = m.group(1)
            out[yyyymm] = f
        return out

    wrote = 0

    for yr in years:
        noah_year_dir = noah_root / f"{yr}"
        vic_year_dir  = vic_root  / f"{yr}"
        out_year_dir  = out_root  / f"{yr}"
        out_year_dir.mkdir(parents=True, exist_ok=True)

        if not noah_year_dir.exists():
            print(f"[WARN] Missing NOAH folder: {noah_year_dir}")
            continue
        if not vic_year_dir.exists():
            print(f"[WARN] Missing VIC folder:  {vic_year_dir}")
            continue

        noah_map = index_files_by_yyyymm(noah_year_dir)
        vic_map  = index_files_by_yyyymm(vic_year_dir)

        # process all 12 months explicitly
        months_written = 0
        for m in range(1, 13):
            yyyymm = f"{yr}{m:02d}"

            if yyyymm not in noah_map:
                print(f"[WARN] {yr}-{m:02d}: NOAH missing")
                continue
            if yyyymm not in vic_map:
                print(f"[WARN] {yr}-{m:02d}: VIC missing")
                continue

            f_noah = noah_map[yyyymm]
            f_vic  = vic_map[yyyymm]

            try:
                ds_noah = xr.open_dataset(f_noah, decode_times=True)
                ds_vic  = xr.open_dataset(f_vic,  decode_times=True)

                if var_name not in ds_noah.variables:
                    raise KeyError(f"{f_noah.name}: variable '{var_name}' not found")
                if var_name not in ds_vic.variables:
                    raise KeyError(f"{f_vic.name}: variable '{var_name}' not found")

                da_noah = ds_noah[var_name]
                da_vic  = ds_vic[var_name]

                # Align grids/coords safely (inner keeps common coords if tiny differences exist)
                da_noah, da_vic = xr.align(da_noah, da_vic, join="inner")

                # Blend
                da_blend = ((da_noah + da_vic) / 2.0) * float(scale_factor)

                # Ensure there is a time dimension (NLDAS monthly usually has time=1)
                if "time" not in da_blend.dims:
                    # create a synthetic time coordinate: first day of that month
                    t = np.datetime64(f"{yr}-{m:02d}-01")
                    da_blend = da_blend.expand_dims(time=[t])

                # Build output dataset
                da_blend.name = out_var_name
                ds_out = da_blend.to_dataset()

                # Metadata
                units = da_noah.attrs.get("units") or ds_noah[var_name].attrs.get("units") or ""
                if units:
                    ds_out[out_var_name].attrs["units"] = units
                ds_out[out_var_name].attrs["blend_method"] = f"(({var_name}_NOAH + {var_name}_VIC)/2) * {scale_factor}"
                ds_out[out_var_name].attrs["source_files"] = f"NOAH={f_noah.name}; VIC={f_vic.name}"

                # Output path in the same year folder structure
                out_nc = out_year_dir / f"{filename_prefix}_{out_var_name}_A{yyyymm}.nc"

                # Compression
                encoding = {out_var_name: {"zlib": True, "complevel": 4}}

                ds_out.to_netcdf(out_nc, encoding=encoding)

                wrote += 1
                months_written += 1
                print(f"✓ wrote {out_nc}")

            except Exception as e:
                print(f"[SKIP] {yr}-{m:02d} -> {e}")

            finally:
                # Close if opened
                try:
                    ds_noah.close()
                except Exception:
                    pass
                try:
                    ds_vic.close()
                except Exception:
                    pass

        if months_written < 12:
            print(f"[INFO] {yr}: wrote {months_written}/12 months (some missing/corrupt)")

    print(f"\nDONE ✅ Total monthly files written: {wrote}")
    print(f"Output root: {out_root}")
    return out_root

### Run the monthly blend writer

Executes the monthly NOAH–VIC blending function over the selected years and writes the blended monthly NetCDF files to disk.


In [ ]:
from pathlib import Path

noah_root = Path(r"T:\Data\Climate_Data\Gridded\Downloaded\NLDAS_2_monthly\Noah\NLDAS\NLDAS_NOAH0125_M.2.0")
vic_root  = Path(r"T:\Data\Climate_Data\Gridded\Downloaded\NLDAS_2_monthly\VIC\NLDAS\NLDAS_VIC0125_M.2.0")
out_root     = Path(r"T:\Data\Climate_Data\Gridded\Downloaded\NLDAS_2_monthly\NOAH_VIC_average")

write_monthly_blend_nc_by_year(
    noah_root=noah_root,
    vic_root=vic_root,
    out_root=out_root,
    years=range(2000, 2026),
    var_name="Qsb",
    scale_factor=0.75
)

### Verbose monthly blend spot-check tools

Defines helper functions to compare one month of NOAH, VIC, and blended output so the blend can be verified before downstream use.


In [ ]:
from pathlib import Path
import re
import xarray as xr
import numpy as np

yyyymm_re = re.compile(r"A(\d{6})")

def index_files_by_yyyymm(year_dir: Path, file_glob="*.nc*"):
    out = {}
    for f in sorted(year_dir.glob(file_glob)):
        m = yyyymm_re.search(f.name)
        if m:
            out[m.group(1)] = f
    return out

def _minmaxmean(a):
    return (float(np.nanmin(a)), float(np.nanmax(a)), float(np.nanmean(a)))

def spot_check_one_month_verbose(
    noah_root: Path,
    vic_root: Path,
    out_root: Path,
    year: int,
    month: int,
    var_name="Qsb",
    out_var_name=None,
    scale_factor=0.7,
    atol=1e-12,
    rtol=1e-6,
):
    if out_var_name is None:
        out_var_name = var_name

    yyyymm = f"{year}{month:02d}"

    noah_map = index_files_by_yyyymm(Path(noah_root) / f"{year}")
    vic_map  = index_files_by_yyyymm(Path(vic_root)  / f"{year}")
    out_map  = index_files_by_yyyymm(Path(out_root)  / f"{year}")

    f_noah = noah_map.get(yyyymm)
    f_vic  = vic_map.get(yyyymm)
    f_out  = out_map.get(yyyymm)

    print("NOAH file:", f_noah)
    print("VIC  file:", f_vic)
    print("OUT  file:", f_out)

    if not (f_noah and f_vic and f_out):
        raise FileNotFoundError("Missing one of NOAH/VIC/OUT files for this month.")

    ds_noah = xr.open_dataset(f_noah)
    ds_vic  = xr.open_dataset(f_vic)
    ds_out  = xr.open_dataset(f_out)

    try:
        da_noah = ds_noah[var_name]
        da_vic  = ds_vic[var_name]
        da_out  = ds_out[out_var_name]

        # Align to compare cell-by-cell
        da_noah, da_vic, da_out = xr.align(da_noah, da_vic, da_out, join="inner")

        expected = ((da_noah + da_vic) / 2.0) * float(scale_factor)

        noah_vals = da_noah.values
        vic_vals  = da_vic.values
        exp_vals  = expected.values
        out_vals  = da_out.values

        nmin, nmax, nmean = _minmaxmean(noah_vals)
        vmin, vmax, vmean = _minmaxmean(vic_vals)
        emin, emax, emean = _minmaxmean(exp_vals)
        omin, omax, omean = _minmaxmean(out_vals)

        print("\n=== VALUE STATS (after alignment) ===")
        print(f"NOAH     min/max/mean: {nmin} / {nmax} / {nmean}")
        print(f"VIC      min/max/mean: {vmin} / {vmax} / {vmean}")
        print(f"EXPECTED min/max/mean: {emin} / {emax} / {emean}   (0.7*(NOAH+VIC)/2)")
        print(f"OUTPUT   min/max/mean: {omin} / {omax} / {omean}   (your saved BLEND file)")

        diff = out_vals - exp_vals
        diff_max_abs  = float(np.nanmax(np.abs(diff)))
        diff_mean_abs = float(np.nanmean(np.abs(diff)))

        denom = np.maximum(np.abs(exp_vals), 1e-30)
        diff_max_rel  = float(np.nanmax(np.abs(diff) / denom))

        print("\n=== CHECK OUTPUT == EXPECTED ===")
        print("diff max_abs :", diff_max_abs)
        print("diff mean_abs:", diff_mean_abs)
        print("diff max_rel :", diff_max_rel)

        ok = np.allclose(out_vals, exp_vals, atol=atol, rtol=rtol, equal_nan=True)
        print("PASS allclose:", ok)

        # Units (helpful)
        print("\n=== UNITS ===")
        print("NOAH units:", ds_noah[var_name].attrs.get("units", ""))
        print("VIC  units:", ds_vic[var_name].attrs.get("units", ""))
        print("OUT  units:", ds_out[out_var_name].attrs.get("units", ""))

        return ok

    finally:
        ds_noah.close()
        ds_vic.close()
        ds_out.close()

### Run a monthly blend spot check

Executes the month-level spot-check for a selected year and month to confirm the blended recharge file was created correctly.


In [ ]:
ok = spot_check_one_month_verbose(
    noah_root=Path(r"T:\Data\Climate_Data\Gridded\Downloaded\NLDAS_2_monthly\Noah\NLDAS\NLDAS_NOAH0125_M.2.0"),
    vic_root=Path(r"T:\Data\Climate_Data\Gridded\Downloaded\NLDAS_2_monthly\VIC\NLDAS\NLDAS_VIC0125_M.2.0"),
    out_root=Path(r"T:\Data\Climate_Data\Gridded\Downloaded\NLDAS_2_monthly\NOAH_VIC_average"),
    year=2000,
    month=8,
    var_name="Qsb",
    scale_factor=0.75
)

### Single-pixel blend verification function

Defines a focused check that compares one pixel among NOAH, VIC, expected blended output, and the written output file.


In [ ]:
import xarray as xr
import numpy as np

def check_one_pixel(f_noah, f_vic, f_out, var="Qsb", scale=0.7, out_var=None):
    if out_var is None:
        out_var = var

    dsN = xr.open_dataset(f_noah)
    dsV = xr.open_dataset(f_vic)
    dsO = xr.open_dataset(f_out)

    daN, daV, daO = xr.align(dsN[var], dsV[var], dsO[out_var], join="inner")

    expected = ((daN + daV) / 2.0) * scale

    # pick the location of the OUTPUT max (so you see why max stats differ)
    vals = daO.values
    idx = np.unravel_index(np.nanargmax(vals), vals.shape)

    n = float(daN.values[idx])
    v = float(daV.values[idx])
    o = float(daO.values[idx])
    e = float(expected.values[idx])

    print("Index of OUTPUT max:", idx)
    print("NOAH @idx:", n)
    print("VIC  @idx:", v)
    print("Expected blend:", 0.7 * (n + v) / 2.0)
    print("Expected (from array):", e)
    print("OUTPUT:", o)
    print("Diff OUTPUT-Expected:", o - e)

    dsN.close(); dsV.close(); dsO.close()

### Run the single-pixel verification

Tests one pixel in one month to make sure the blended NetCDF value matches the intended averaging and scaling rule.


In [ ]:
check_one_pixel(
    f_noah=r"T:\Data\Climate_Data\Gridded\Downloaded\NLDAS_2_monthly\Noah\NLDAS\NLDAS_NOAH0125_M.2.0\2019\NLDAS_NOAH0125_M.A201908.020.nc",
    f_vic =r"T:\Data\Climate_Data\Gridded\Downloaded\NLDAS_2_monthly\VIC\NLDAS\NLDAS_VIC0125_M.2.0\2019\NLDAS_VIC0125_M.A201908.020.nc",
    f_out =r"T:\Data\Climate_Data\Gridded\Downloaded\NLDAS_2_monthly\NOAH_VIC_average\2019\BLEND_Qsb_A201908.nc",
    var="Qsb",
    scale=0.75
)

### Inspect blended NetCDF metadata

Opens one blended NetCDF file and prints dataset and variable metadata so units and attributes can be confirmed.


In [ ]:
import xarray as xr
nc_path = r"T:\Data\Climate_Data\Gridded\Downloaded\NLDAS_2_monthly\NOAH_VIC_average\NLDAS\NLDAS_NOAHVIC_M.2.0\2000\BLEND_Qsb_A200001.nc"
ds = xr.open_dataset(nc_path, engine="netcdf4", decode_times=False)
print(ds["Qsb"].attrs)
print(ds.attrs)

### Inspect blended variable units and ranges

Prints a few key attributes and min/max values from the blended recharge variable for a quick sanity check.


In [ ]:
da = ds["Qsb"]
print("units:", da.attrs.get("units"))
print("cell_methods:", da.attrs.get("cell_methods"))
print("long_name:", da.attrs.get("long_name"))
print("min/max:", float(np.nanmin(da.values)), float(np.nanmax(da.values)))

### Convert annual recharge NetCDF to MODFLOW-ready rasters

Reads the annual recharge NetCDF, converts the chosen units to m/day, clips to the basin boundary, and writes MODFLOW-grid recharge rasters.


In [ ]:
import arcpy
import os
import numpy as np
import xarray as xr
from arcpy.sa import ExtractByMask, Con, IsNull, Raster

arcpy.CheckOutExtension("Spatial")
arcpy.env.overwriteOutput = True
arcpy.env.parallelProcessingFactor = "100%"
arcpy.env.addOutputsToMap = False

# ============================================================
# INPUTS
# ============================================================
nc_file   = r"D:\Users\abolmaal\modelling\Projects\GreatLakes\outputs\NOAH-VIC_annual_recharge_2000_2025.nc"
varname   = "Qsb"
x_dim     = "lon"
y_dim     = "lat"
t_dim     = "year"

boundary_shp = r"S:\Projects\Active\GLB_LHM\LHM_inputs\Boundary\extendedBdry_jan26_adk.shp"

out_dir = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\Recharge"
os.makedirs(out_dir, exist_ok=True)

CELL_SIZE_OUT = 1000  # meters (match your MF6 grid)

# ============================================================
# UNIT CONVERSION TO m/day  (choose correct mode)
# ============================================================
# Your NetCDF max ~252 strongly suggests mm/year:
# m/day = (mm/yr)/1000/365.25
CONV_MODE = "mm_per_year"   # alternatives: "kg_m2_s", "m_per_s", "m_per_day"

def to_mday(r):
    if CONV_MODE == "mm_per_year":
        return r / 365.2595636
    if CONV_MODE == "kg_m2_s":
        return (r * 86400.0) / 1000.0
    if CONV_MODE == "m_per_s":
        return r * 86400.0
    if CONV_MODE == "m_per_day":
        return r
    raise ValueError("Bad CONV_MODE")

# ============================================================
# SPATIAL REFERENCES
# ============================================================
sr_wgs84 = arcpy.SpatialReference(4326)
sr_3174  = arcpy.SpatialReference(3174)

tx_list = arcpy.ListTransformations(sr_wgs84, sr_3174)
geo_tx = tx_list[0] if tx_list else ""
print("Geographic transformation:", geo_tx if geo_tx else "(none listed)")

# ============================================================
# SCRATCH WORKSPACE (avoid in_memory for NetCDF workflows)
# ============================================================
scratch_gdb = os.path.join(out_dir, "rech_scratch.gdb")
if not arcpy.Exists(scratch_gdb):
    arcpy.management.CreateFileGDB(out_dir, os.path.basename(scratch_gdb))

arcpy.env.workspace = scratch_gdb
arcpy.env.scratchWorkspace = scratch_gdb

# ============================================================
# 0) NetCDF sanity check
# ============================================================
ds = xr.open_dataset(nc_file)
years = [int(y) for y in ds[t_dim].values]
a0 = ds[varname].isel({t_dim: 0}).values
print(f"NetCDF check (year={years[0]}): min/max = {float(np.nanmin(a0))} / {float(np.nanmax(a0))}")

# ============================================================
# 1) Boundary prep (make sure it is EPSG:3174)
# ============================================================
print("\n1/5 Boundary prep...")
boundary_3174 = os.path.join(out_dir, "boundary_3174.shp")
if arcpy.Exists(boundary_3174):
    arcpy.management.Delete(boundary_3174)

b_sr = arcpy.Describe(boundary_shp).spatialReference
if b_sr and b_sr.factoryCode == 3174:
    arcpy.management.CopyFeatures(boundary_shp, boundary_3174)
else:
    # project if not already 3174
    arcpy.management.Project(boundary_shp, boundary_3174, sr_3174)

bd = arcpy.Describe(boundary_3174)
print("  boundary_3174 SR:", bd.spatialReference.name, bd.spatialReference.factoryCode)
print("  boundary_3174 extent:", (bd.extent.XMin, bd.extent.YMin, bd.extent.XMax, bd.extent.YMax))

# ============================================================
# 2) Output folder for yearly rasters
# ============================================================
print("\n2/5 Output folder...")
year_dir = os.path.join(out_dir, f"rech_mday_{CELL_SIZE_OUT}m")
os.makedirs(year_dir, exist_ok=True)
print("  ✅", year_dir)

# ============================================================
# Helper: safe gp messages
# ============================================================
def gp_fail(tag):
    print(f"\n❌ FAILED at: {tag}")
    print(arcpy.GetMessages(2))
    raise arcpy.ExecuteError(arcpy.GetMessages(2))

def raster_max(path):
    return float(arcpy.management.GetRasterProperties(path, "MAXIMUM").getOutput(0))

# ============================================================
# 3) Loop years (BY_INDEX) -> ProjectRaster -> Mask -> Convert -> Write GeoTIFF
# ============================================================
print("\n3/5 Build yearly rasters (m/day)...")

for idx, y in enumerate(years, start=1):
    print(f"\n  Year {y} (index {idx})...")

    # ---- IMPORTANT: clear extent while projecting from WGS84 NetCDF ----
    arcpy.ClearEnvironment("extent")
    arcpy.ClearEnvironment("cellSize")
    arcpy.ClearEnvironment("snapRaster")
    arcpy.env.outputCoordinateSystem = None

    lyr_name = f"Qsb_{y}_lyr"
    try:
        res = arcpy.md.MakeNetCDFRasterLayer(
            in_netCDF_file=nc_file,
            variable=varname,
            x_dimension=x_dim,
            y_dimension=y_dim,
            out_raster_layer=lyr_name,
            band_dimension="",
            dimension_values=f"{t_dim} {idx}",
            value_selection_method="BY_INDEX"
        )
        nc_lyr = res.getOutput(0)  # robust handle
    except arcpy.ExecuteError:
        gp_fail("MakeNetCDFRasterLayer")

    # Project directly from NetCDF layer (NO CopyRaster)
    tmp_3174 = os.path.join(scratch_gdb, f"qsb_{y}_3174")
    if arcpy.Exists(tmp_3174):
        arcpy.management.Delete(tmp_3174)

    try:
        arcpy.management.ProjectRaster(
            in_raster=nc_lyr,
            out_raster=tmp_3174,
            out_coor_system=sr_3174,
            resampling_type="BILINEAR",
            cell_size=CELL_SIZE_OUT,
            geographic_transform=geo_tx,
            in_coor_system=sr_wgs84  # force input CRS
        )
    except arcpy.ExecuteError:
        gp_fail("ProjectRaster (NetCDF->3174)")

    # Now we can set processing env to boundary (same CRS)
    arcpy.env.outputCoordinateSystem = sr_3174
    arcpy.env.extent = boundary_3174
    arcpy.env.cellSize = CELL_SIZE_OUT

    # Mask to boundary
    try:
        masked = ExtractByMask(Raster(tmp_3174), boundary_3174)
    except arcpy.ExecuteError:
        gp_fail("ExtractByMask")

    # Save raw masked (debug + ensures stats exist)
    raw_tif = os.path.join(year_dir, f"Qsb_raw_{y}.tif")
    if arcpy.Exists(raw_tif):
        arcpy.management.Delete(raw_tif)

    try:
        masked.save(raw_tif)
        arcpy.management.DefineProjection(raw_tif, sr_3174)
        arcpy.management.CalculateStatistics(raw_tif)
    except arcpy.ExecuteError:
        gp_fail("Save raw masked TIFF")

    mx_raw = raster_max(raw_tif)
    print("    raw MAX:", mx_raw)

    if idx == 1 and mx_raw == 0.0:
        # If this happens, it means overlap/projection is still wrong
        r_ext = arcpy.Describe(raw_tif).extent
        b_ext = arcpy.Describe(boundary_3174).extent
        raise RuntimeError(
            "Year 2000 raw MAX is 0 after masking.\n"
            f"raw extent: {(r_ext.XMin,r_ext.YMin,r_ext.XMax,r_ext.YMax)}\n"
            f"bdry extent: {(b_ext.XMin,b_ext.YMin,b_ext.XMax,b_ext.YMax)}\n"
            "This indicates the projected raster and boundary do not overlap."
        )

    # Convert NoData->0 then to m/day
    masked0 = Con(IsNull(Raster(raw_tif)), 0, Raster(raw_tif))
    r_mday = to_mday(masked0)
    r_mday = Con(r_mday < 0, 0, r_mday)

    # Save m/day to scratch first (keeps CRS), then ProjectRaster -> GeoTIFF (embeds CRS tags)
    mday_tmp = os.path.join(scratch_gdb, f"rech_mday_{y}")
    if arcpy.Exists(mday_tmp):
        arcpy.management.Delete(mday_tmp)

    try:
        r_mday.save(mday_tmp)
        arcpy.management.DefineProjection(mday_tmp, sr_3174)
    except arcpy.ExecuteError:
        gp_fail("Save m/day (scratch)")

    out_tif = os.path.join(year_dir, f"rech_mday_{y}.tif")
    if arcpy.Exists(out_tif):
        arcpy.management.Delete(out_tif)

    try:
        arcpy.management.ProjectRaster(
            in_raster=mday_tmp,
            out_raster=out_tif,
            out_coor_system=sr_3174,
            resampling_type="NEAREST",
            cell_size=CELL_SIZE_OUT,
            in_coor_system=sr_3174
        )
        arcpy.management.CalculateStatistics(out_tif)
    except arcpy.ExecuteError:
        gp_fail("ProjectRaster (mday->GeoTIFF)")

    print("    m/day MAX:", raster_max(out_tif))

print("\nDONE ✅")
print("Yearly recharge rasters (m/day):", year_dir)
print("Conversion mode:", CONV_MODE)


# 7.  Drains
Build the cross-border stream network and prepare it for drain or stream-package inputs.


### 7.1 Streams

#### Extract and clip NHD streams

Builds a clipped stream network from the National Hydrography Dataset inside the model boundary. This is the first step toward drain/stream inputs.


In [ ]:
import os
import arcpy

arcpy.env.overwriteOutput = True

# ----------------------------
# INPUTS
# ----------------------------
nhd_gdb  = r"S:\Data\GIS_Data\Downloaded\NHD\NHD_H_National_GDB.gdb"
boundary = r"S:\Projects\Active\GLB_LHM\LHM_inputs\Boundary\extendedBdry_jan26_adk.shp"
out_dir  = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\NHD"
os.makedirs(out_dir, exist_ok=True)

# ----------------------------
# OUTPUT + SCRATCH GDBs
# ----------------------------
out_gdb = os.path.join(out_dir, "NHD_streams.gdb")
if not arcpy.Exists(out_gdb):
    arcpy.management.CreateFileGDB(out_dir, os.path.basename(out_gdb))

scratch_gdb = os.path.join(out_dir, "_scratch_nhd.gdb")
if not arcpy.Exists(scratch_gdb):
    arcpy.management.CreateFileGDB(out_dir, os.path.basename(scratch_gdb))

# ----------------------------
# FIND NHDFlowline
# ----------------------------
candidates = [
    os.path.join(nhd_gdb, "Hydrography", "NHDFlowline"),
    os.path.join(nhd_gdb, "NHDFlowline"),
]
flow_fc = next((p for p in candidates if arcpy.Exists(p)), None)
if flow_fc is None:
    raise FileNotFoundError("Could not find NHDFlowline in the GDB. Tried:\n" + "\n".join(candidates))

print("Using flowlines:", flow_fc)

# ----------------------------
# Get StreamRiver code from domain if possible
# (fallback to 460 which is StreamRiver in NHD)
# ----------------------------
def get_streamriver_code(gdb_path, fc_path, field_name="FType", label="StreamRiver", fallback=460):
    # find field (case-insensitive)
    fld = next((f for f in arcpy.ListFields(fc_path) if f.name.lower() == field_name.lower()), None)
    if fld is None:
        raise ValueError(f"Field '{field_name}' not found in {fc_path}")

    dom_name = fld.domain
    if not dom_name:
        return fallback  # may be coded but no domain exposed; use fallback

    for dom in arcpy.da.ListDomains(gdb_path):
        if dom.name == dom_name and hasattr(dom, "codedValues"):
            for code, lab in dom.codedValues.items():
                if str(lab).strip().lower() == label.lower():
                    return code
    return fallback

stream_code = get_streamriver_code(nhd_gdb, flow_fc, "FType", "StreamRiver", fallback=460)
print("StreamRiver code:", stream_code)

# Build SQL where clause based on field type
ftype_field = next((f for f in arcpy.ListFields(flow_fc) if f.name.lower() == "ftype"), None)
if ftype_field is None:
    raise ValueError("FType field not found (unexpected).")

fld_delim = arcpy.AddFieldDelimiters(flow_fc, ftype_field.name)

if ftype_field.type in ("Integer", "SmallInteger", "Double", "Single"):
    where = f"{fld_delim} = {int(stream_code)}"
else:
    where = f"{fld_delim} = 'StreamRiver'"

print("Where clause:", where)

# ----------------------------
# Project boundary to flowlines CRS (for reliable Select By Location)
# ----------------------------
sr_flow = arcpy.Describe(flow_fc).spatialReference
sr_bdry = arcpy.Describe(boundary).spatialReference

boundary_proj = os.path.join(scratch_gdb, "boundary_proj_to_flow")
if sr_bdry and sr_flow and sr_bdry.factoryCode != sr_flow.factoryCode:
    arcpy.management.Project(boundary, boundary_proj, sr_flow)
    boundary_for_select = boundary_proj
    print(f"Projected boundary -> flow SR (EPSG:{sr_flow.factoryCode}) for selection")
else:
    boundary_for_select = boundary

# ----------------------------
# Make StreamRiver layer + select by location
# ----------------------------
lyr = "nhd_streamriver_lyr"
arcpy.management.MakeFeatureLayer(flow_fc, lyr, where)

cnt_all = int(arcpy.management.GetCount(lyr)[0])
print("StreamRiver features (nationwide, before spatial select):", cnt_all)

arcpy.management.SelectLayerByLocation(
    in_layer=lyr,
    overlap_type="INTERSECT",
    select_features=boundary_for_select,
    selection_type="NEW_SELECTION"
)

cnt_sel = int(arcpy.management.GetCount(lyr)[0])
print("Selected StreamRiver features intersecting boundary:", cnt_sel)

if cnt_sel == 0:
    raise RuntimeError(
        "0 features selected. Likely CRS/extent mismatch or boundary not overlapping NHDFlowline. "
        "Check boundary location and projection."
    )

# ----------------------------
# Copy selected features (now guaranteed to exist)
# ----------------------------
temp_sel = os.path.join(scratch_gdb, "NHD_StreamRiver_sel")
arcpy.management.CopyFeatures(lyr, temp_sel)

if not arcpy.Exists(temp_sel):
    raise RuntimeError("CopyFeatures did not create temp_sel (unexpected).")

# ----------------------------
# Project selected streams to boundary CRS (so Clip is clean)
# ----------------------------
temp_sel_proj = os.path.join(scratch_gdb, "NHD_StreamRiver_sel_proj_to_bdry")
use_fc = temp_sel

if sr_bdry and sr_flow and sr_bdry.factoryCode != sr_flow.factoryCode:
    arcpy.management.Project(temp_sel, temp_sel_proj, sr_bdry)
    use_fc = temp_sel_proj
    print(f"Projected selected streams -> boundary SR (EPSG:{sr_bdry.factoryCode})")

# ----------------------------
# Clip to boundary
# ----------------------------
out_fc = os.path.join(out_gdb, "NHDFlowline_StreamRiver_clip")
arcpy.analysis.PairwiseClip(use_fc, boundary, out_fc)  # faster than Clip in many cases

print("DONE ✅")
print("Output:", out_fc)


#### Merge NHD and Ontario hydrography

Merges U.S. NHD flowlines with Ontario hydrography, removes overlaps, and creates one cross-border stream network for the basin.


In [ ]:
import os
import arcpy

arcpy.env.overwriteOutput = True
arcpy.env.parallelProcessingFactor = "0"   # stability

# -----------------------------
# INPUTS
# -----------------------------
nhd_fc = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\NHD\NHD_streams.gdb\NHDFlowline_StreamRiver_clip"
ohn_fc = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\NHD\Ontario_Hydro_Network_(OHN)_-_Watercourse.shp"
boundary = r"S:\Projects\Active\GLB_LHM\LHM_inputs\Boundary\extendedBdry_jan26_adk.shp"

out_dir = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\NHD\_merged_streams_out"
os.makedirs(out_dir, exist_ok=True)

out_gdb = os.path.join(out_dir, "streams_merge_fast.gdb")
if not arcpy.Exists(out_gdb):
    arcpy.management.CreateFileGDB(out_dir, os.path.basename(out_gdb))

# overlap tolerance (meters): how close OHN can be to NHD to be considered duplicate
tol_m = 25  # try 10–50

def _del_if_exists(p):
    if arcpy.Exists(p):
        arcpy.management.Delete(p)

def _pairwise_clip(in_fc, clip_fc, out_fc):
    _del_if_exists(out_fc)
    try:
        arcpy.analysis.PairwiseClip(in_fc, clip_fc, out_fc)
    except Exception:
        arcpy.analysis.Clip(in_fc, clip_fc, out_fc)
    return out_fc

# -----------------------------
# 1) Get target CRS from boundary
# -----------------------------
target_sr = arcpy.Describe(boundary).spatialReference
if target_sr is None:
    raise RuntimeError("Boundary has no spatial reference. Define projection first.")

# -----------------------------
# 2) Ensure OHN is in boundary CRS (project only OHN)
# -----------------------------
ohn_prj = os.path.join(out_gdb, "ohn_prj")
sr_ohn = arcpy.Describe(ohn_fc).spatialReference

if sr_ohn and target_sr and sr_ohn.factoryCode == target_sr.factoryCode:
    ohn_use = ohn_fc
else:
    _del_if_exists(ohn_prj)
    arcpy.management.Project(ohn_fc, ohn_prj, target_sr)
    ohn_use = ohn_prj

# -----------------------------
# 3) Clip OHN to boundary (NHD already clipped, so we keep it as-is)
# -----------------------------
ohn_clip = os.path.join(out_gdb, "ohn_clip_bdry")
_pairwise_clip(ohn_use, boundary, ohn_clip)

# Clean + index (helps speed)
arcpy.management.RepairGeometry(ohn_clip)
try:
    arcpy.management.AddSpatialIndex(ohn_clip)
except Exception:
    pass
try:
    arcpy.management.AddSpatialIndex(nhd_fc)  # harmless; may already exist
except Exception:
    pass

# -----------------------------
# 4) Remove OHN overlaps: drop OHN features within tol_m of NHD
#    (keeps NHD priority)
# -----------------------------
ohn_keep = os.path.join(out_gdb, "ohn_keep_not_near_nhd")
_del_if_exists(ohn_keep)

nhd_lyr = "nhd_lyr"
ohn_lyr = "ohn_lyr"
arcpy.management.MakeFeatureLayer(nhd_fc, nhd_lyr)
arcpy.management.MakeFeatureLayer(ohn_clip, ohn_lyr)

# Select OHN features that are near/overlapping NHD
arcpy.management.SelectLayerByLocation(
    in_layer=ohn_lyr,
    overlap_type="WITHIN_A_DISTANCE",
    select_features=nhd_lyr,
    search_distance=f"{tol_m} Meters",
    selection_type="NEW_SELECTION"
)

# Keep only OHN features NOT selected
arcpy.management.SelectLayerByAttribute(ohn_lyr, "SWITCH_SELECTION")
arcpy.management.CopyFeatures(ohn_lyr, ohn_keep)

# -----------------------------
# 5) Merge final
# -----------------------------
merged_streams = os.path.join(out_gdb, "streams_merged_noOverlap")
_del_if_exists(merged_streams)
arcpy.management.Merge([nhd_fc, ohn_keep], merged_streams)

# Optional: remove exact duplicates (cheap)
try:
    arcpy.management.DeleteIdentical(merged_streams, ["Shape"], xy_tolerance=f"{tol_m} Meters")
except Exception:
    pass

arcpy.management.RepairGeometry(merged_streams)

print("DONE ✅")
print("Final merged streams:", merged_streams)
print("OHN clipped:", ohn_clip)
print("OHN kept (not near NHD):", ohn_keep)

#### Open merged streams in GeoPandas

Loads the merged stream layer into a GeoDataFrame so the attributes can be inspected before assigning widths or exporting.


In [ ]:
import os
import arcpy
import geopandas as gpd

arcpy.env.overwriteOutput = True
arcpy.management.ClearWorkspaceCache()

gdb = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\NHD\_merged_streams_out\streams_merge_fast.gdb"
arcpy.env.workspace = gdb

in_fc = "streams_merged_noOverlap"   # <-- use the name, not the full path

gdf = gpd.read_file(gdb, layer=in_fc, driver="OpenFileGDB")
print(gdf.head())

#### Export cleaned merged streams

Writes the merged stream GeoDataFrame back to a shapefile for later drain/stream processing.


In [ ]:
# copy the geodataframe back to a new shapefile (or feature class) with only selected fields
out_dir = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\NHD"

out_shp = os.path.join(out_dir, "streams_merged_clip_to_bdry.shp")
if arcpy.Exists(out_shp):
    arcpy.management.Delete(out_shp)
gdf.to_file(out_shp, driver="ESRI Shapefile")
print("✅ Exported to shapefile:", out_shp)

#### Define stream-width rules

Creates a width-classification rule based on available NHD or Ontario attributes. This is used to assign reasonable channel widths where measured widths are unavailable.


In [ ]:
import numpy as np

def width_rule(row):
    # NHD: FCode + GNIS_NAME
    fcode = row.get("FCode", None)
    gnis  = row.get("GNIS_NAME", None)

    if fcode is not None:
        try:
            fcode = int(fcode)
        except:
            fcode = None

        if fcode == 46003:      # ditch/canal
            w = 4.0
        elif fcode == 46000:    # stream/river
            w = 10.0
        else:
            w = 10.0

        if isinstance(gnis, str) and gnis.strip() != "":
            w = max(w, 25.0)
        return w

    # OHN fallback: FLOW_CLASS / WATERCOURS / PERMANENCY
    wc = str(row.get("WATERCOURS", "")).lower()
    cl = str(row.get("FLOW_CLASS", "")).lower()
    perm = str(row.get("PERMANENCY", "")).lower()

    if "river" in wc:
        w = 25.0
    elif "stream" in wc:
        w = 10.0
    elif "primary" in cl:
        w = 10.0
    elif "secondary" in cl:
        w = 6.0
    else:
        w = 5.0

    if perm and (not perm.startswith("perm")):
        w *= 0.7

    return float(np.clip(w, 1.0, 300.0))

gdf["Width_m"] = gdf.apply(width_rule, axis=1)
print(gdf["Width_m"].describe())

#### Apply stream widths to the merged network

Reads the merged stream shapefile, finds the relevant fields even if names were truncated, and computes width values for the stream features.


In [ ]:
import os, glob
import geopandas as gpd
import numpy as np

shp = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\NHD\streams_merged_clip_to_bdry.shp"

# --- read shapefile ---
gdf = gpd.read_file(shp)

# --- helper to find fields even if names got truncated ---
cols = list(gdf.columns)
lut = {c.lower(): c for c in cols}

def pick(candidates):
    for c in candidates:
        if c.lower() in lut:
            return lut[c.lower()]
    return None

F_FCODE = pick(["FCode", "FCODE"])
F_GNIS  = pick(["GNIS_NAME", "GNISNAME"])
F_WATER = pick(["WATERCOURS", "WATERCOUR", "WATERCOURSE"])
F_CLASS = pick(["FLOW_CLASS", "FLOWCLAS"])
F_PERM  = pick(["PERMANENCY", "PERMANENC"])

print("Using fields:", F_FCODE, F_GNIS, F_WATER, F_CLASS, F_PERM)

# --- width rule ---
def width_rule(row):
    w = None

    # NHD logic if FCode exists on that feature
    fcode = row.get(F_FCODE) if F_FCODE else None
    gnis  = row.get(F_GNIS)  if F_GNIS  else None

    if fcode is not None and str(fcode) != "":
        try:
            fcode_i = int(fcode)
        except Exception:
            fcode_i = None

        if fcode_i == 46003:      # ditch/canal
            w = 4.0
        elif fcode_i == 46000:    # stream/river
            w = 10.0
        else:
            w = 10.0

        if isinstance(gnis, str) and gnis.strip() != "":
            w = max(w, 25.0)

    # OHN fallback if no FCode
    if w is None:
        wc = str(row.get(F_WATER, "")).lower() if F_WATER else ""
        cl = str(row.get(F_CLASS, "")).lower() if F_CLASS else ""
        pm = str(row.get(F_PERM,  "")).lower() if F_PERM  else ""

        if "river" in wc:
            w = 25.0
        elif "stream" in wc:
            w = 10.0
        elif "primary" in cl:
            w = 10.0
        elif "secondary" in cl:
            w = 6.0
        else:
            w = 5.0

        if pm and (not pm.startswith("perm")):
            w *= 0.7

    return float(np.clip(w, 1.0, 300.0))

# --- add shapefile-safe width field ---
gdf["WIDTH_M"] = gdf.apply(width_rule, axis=1).astype("float64")

# --- OPTIONAL but recommended: keep only a few columns to avoid 2GB DBF ---
keep = ["geometry", "WIDTH_M"]
for f in [F_FCODE, F_GNIS, F_WATER, F_CLASS, F_PERM]:
    if f and f not in keep:
        keep.append(f)

gdf_out = gdf[keep].copy()

# --- delete existing shapefile parts so overwrite is clean ---
base = os.path.splitext(shp)[0]
for p in glob.glob(base + ".*"):
    try:
        os.remove(p)
    except Exception:
        pass

# --- write back to the SAME shapefile path ---
gdf_out.to_file(shp, driver="ESRI Shapefile")
print("✅ Wrote WIDTH_M back to:", shp)
print("Saved columns:", list(gdf_out.columns))

#### Set the drain-input stream path

Stores the final stream shapefile path used by the downstream drain-preparation cell.


In [ ]:
pathInputDrn = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\NHD\streams_merged_clip_to_bdry.shp"

#### Project and align streams to the MODFLOW grid

Projects the stream network to the model grid CRS, clips it, and prepares grid-aligned stream data for later raster or boundary-package use.


In [ ]:
import arcpy, os

arcpy.env.overwriteOutput = True

in_streams = pathInputDrn
template = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\GRID_3174\template_5000m_epsg3174.tif"

out_dir = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\NHD"
os.makedirs(out_dir, exist_ok=True)

gdb = os.path.join(out_dir, "streams_tmp.gdb")
if not arcpy.Exists(gdb):
    arcpy.management.CreateFileGDB(out_dir, "streams_tmp.gdb")

proj_fc = os.path.join(gdb, "streams_3174")
clip_fc = os.path.join(gdb, "streams_3174_clip")

# spatial reference from template
sr = arcpy.Describe(template).spatialReference

# --- Project ---
if arcpy.Exists(proj_fc):
    arcpy.management.Delete(proj_fc)

arcpy.management.Project(in_streams, proj_fc, sr)
print("Projected to:", proj_fc, "exists?", arcpy.Exists(proj_fc))

# --- Build extent polygon feature class in the SAME GDB ---
ext = arcpy.Describe(template).extent
extent_poly = arcpy.Polygon(arcpy.Array([
    arcpy.Point(ext.XMin, ext.YMin),
    arcpy.Point(ext.XMin, ext.YMax),
    arcpy.Point(ext.XMax, ext.YMax),
    arcpy.Point(ext.XMax, ext.YMin),
    arcpy.Point(ext.XMin, ext.YMin),
]), sr)

extent_fc = os.path.join(gdb, "template_extent")
if arcpy.Exists(extent_fc):
    arcpy.management.Delete(extent_fc)
arcpy.management.CopyFeatures(extent_poly, extent_fc)

# --- Clip ---
if arcpy.Exists(clip_fc):
    arcpy.management.Delete(clip_fc)

arcpy.analysis.Clip(proj_fc, extent_fc, clip_fc)
print("Clipped to:", clip_fc, "exists?", arcpy.Exists(clip_fc))

# Optional: export clipped to shapefile (for geopandas)
clip_shp = os.path.join(out_dir, "streams_3174_clip_to_modelgrid.shp")
if arcpy.Exists(clip_shp):
    arcpy.management.Delete(clip_shp)
arcpy.conversion.FeatureClassToFeatureClass(clip_fc, out_dir, "streams_3174_clip_to_modelgrid.shp")

print("✅ clipped shapefile:", clip_shp, "exists?", arcpy.Exists(clip_shp))

### 7.2 Wetlands

In [ ]:
# Define the path to all the Great lakes wetlands 
# US side from national wetland inventory
NY = r"S:\Data\GIS_Data\Downloaded\National_Wetlands_Inventory\NY_geodatabase_wetlands.gdb\NY_Wetlands"
PA = r"S:\Data\GIS_Data\Downloaded\National_Wetlands_Inventory\PA_geodatabase_wetlands.gdb\PA_Wetlands"
OH = r"S:\Data\GIS_Data\Downloaded\National_Wetlands_Inventory\OH_geodatabase_wetlands.gdb\OH_Wetlands"
MN =r"S:\Data\GIS_Data\Downloaded\National_Wetlands_Inventory\MN_geodatabase_wetlands.gdb\MN_Wetlands"
WI = r"S:\Data\GIS_Data\Downloaded\National_Wetlands_Inventory\WI_geodatabase_wetlands.gdb\WI_Wetlands"
MI = r"S:\Data\GIS_Data\Downloaded\National_Wetlands_Inventory\MI_Wetlands_Geopackage.gpkg\main.MI_Wetlands"
IN = r"S:\Data\GIS_Data\Downloaded\National_Wetlands_Inventory\IN_geodatabase_wetlands.gdb\IN_Wetlands"
IL = r"S:\Data\GIS_Data\Downloaded\National_Wetlands_Inventory\IL_geodatabase_wetlands.gdb\IL_Wetlands"

# Ontarion Canadas wetlands
CN = r"S:\Data\GIS_Data\Downloaded\Canada\CNWI\INTHC_CNWI_ON.gdb\INTHC_CNWI_ON.gdb\INTHC_CNWI_ON.gdb\INTHC_CNWI_ON"


# merge all the following wetlands and clip it to the GL boundary
GL_Bdr = r"S:\Projects\Active\GLB_LHM\LHM_inputs\Boundary\extendedBdry_jan26_adk.shp"
# also make sure if wetlands overlap with the lake remove the part of the wetlands is in the lake 
Greatlakes = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\Lakes\GreatLakes.shp"


# save the wetlands in the following directory
Wetlands_GL = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\Wetlands"

In [ ]:
import os
import arcpy

arcpy.env.overwriteOutput = True
arcpy.env.parallelProcessingFactor = "100%"

# ============================================================
# INPUTS
# ============================================================
# US side from National Wetlands Inventory
NY = r"S:\Data\GIS_Data\Downloaded\National_Wetlands_Inventory\NY_geodatabase_wetlands.gdb\NY_Wetlands"
PA = r"S:\Data\GIS_Data\Downloaded\National_Wetlands_Inventory\PA_geodatabase_wetlands.gdb\PA_Wetlands"
OH = r"S:\Data\GIS_Data\Downloaded\National_Wetlands_Inventory\OH_geodatabase_wetlands.gdb\OH_Wetlands"
MN = r"S:\Data\GIS_Data\Downloaded\National_Wetlands_Inventory\MN_geodatabase_wetlands.gdb\MN_Wetlands"
WI = r"S:\Data\GIS_Data\Downloaded\National_Wetlands_Inventory\WI_geodatabase_wetlands.gdb\WI_Wetlands"
MI = r"S:\Data\GIS_Data\Downloaded\National_Wetlands_Inventory\MI_Wetlands_Geopackage.gpkg\main.MI_Wetlands"
IN = r"S:\Data\GIS_Data\Downloaded\National_Wetlands_Inventory\IN_geodatabase_wetlands.gdb\IN_Wetlands"
IL = r"S:\Data\GIS_Data\Downloaded\National_Wetlands_Inventory\IL_geodatabase_wetlands.gdb\IL_Wetlands"

# Ontario / Canada wetlands
CN = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\Wetlands\INTHC_CNWI_ON.gdb\INTHC_CNWI_ON"  
# ^^^ adjust this if needed; your original path looks duplicated

# Boundary and lakes
GL_Bdr = r"S:\Projects\Active\GLB_LHM\LHM_inputs\Boundary\extendedBdry_jan26_adk.shp"
GreatLakes = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\Lakes\GreatLakes.shp"

# Output folder
Wetlands_GL = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\Wetlands"

# ============================================================
# USER SETTINGS
# ============================================================
final_gdb_name = "wetlands_processing.gdb"
final_fc_name  = "Wetlands_GL_NoLakes"
export_shp     = True   # optional shapefile export at end
do_dissolve    = False  # set True only if you want one dissolved wetland layer

# ============================================================
# HELPERS
# ============================================================
def msg(txt):
    print(txt)
    arcpy.AddMessage(txt)

def ensure_folder(path):
    if not os.path.isdir(path):
        os.makedirs(path)

def ensure_gdb(folder, gdb_name):
    gdb = os.path.join(folder, gdb_name)
    if not arcpy.Exists(gdb):
        arcpy.management.CreateFileGDB(folder, gdb_name)
    return gdb

def safe_delete(path):
    try:
        if arcpy.Exists(path):
            arcpy.management.Delete(path)
    except Exception:
        pass

def get_count(fc):
    return int(arcpy.management.GetCount(fc)[0])

def has_field(fc, field_name):
    return field_name.lower() in [f.name.lower() for f in arcpy.ListFields(fc)]

def project_if_needed(in_fc, out_fc, target_sr):
    in_sr = arcpy.Describe(in_fc).spatialReference
    if in_sr.name == target_sr.name and in_sr.factoryCode == target_sr.factoryCode:
        arcpy.management.CopyFeatures(in_fc, out_fc)
    else:
        arcpy.management.Project(in_fc, out_fc, target_sr)

# ============================================================
# PREP OUTPUTS
# ============================================================
ensure_folder(Wetlands_GL)
work_gdb = ensure_gdb(Wetlands_GL, final_gdb_name)

boundary_proj = os.path.join(work_gdb, "boundary_proj")
lakes_proj    = os.path.join(work_gdb, "greatlakes_proj")
final_fc      = os.path.join(work_gdb, final_fc_name)

# cleanup old intermediates/final
for p in [boundary_proj, lakes_proj, final_fc]:
    safe_delete(p)

# ============================================================
# CHECK INPUTS
# ============================================================
wetland_inputs = {
    "NY": NY,
    "PA": PA,
    "OH": OH,
    "MN": MN,
    "WI": WI,
    "MI": MI,
    "IN": IN,
    "IL": IL,
    "CN": CN,
}

msg("Checking inputs...")
missing = []
for k, v in wetland_inputs.items():
    if not arcpy.Exists(v):
        missing.append((k, v))

if not arcpy.Exists(GL_Bdr):
    raise FileNotFoundError(f"Boundary not found:\n{GL_Bdr}")

if not arcpy.Exists(GreatLakes):
    raise FileNotFoundError(f"Great Lakes layer not found:\n{GreatLakes}")

if missing:
    msg("The following wetland inputs were not found:")
    for k, v in missing:
        msg(f"  {k}: {v}")
    raise FileNotFoundError("Fix missing wetland paths before running.")

msg("All required inputs found.")

# ============================================================
# PROJECT BOUNDARY + LAKES ONCE
# ============================================================
msg("\nProjecting boundary and lakes...")
target_sr = arcpy.Describe(GL_Bdr).spatialReference
msg(f"Target CRS: {target_sr.name} (EPSG: {target_sr.factoryCode})")

arcpy.management.CopyFeatures(GL_Bdr, boundary_proj)
project_if_needed(GreatLakes, lakes_proj, target_sr)

arcpy.management.RepairGeometry(boundary_proj, "DELETE_NULL")
arcpy.management.RepairGeometry(lakes_proj, "DELETE_NULL")

# ============================================================
# CREATE EMPTY FINAL FEATURE CLASS
# Keep schema very small for speed and size
# ============================================================
msg("\nCreating final output feature class...")
arcpy.management.CreateFeatureclass(
    out_path=work_gdb,
    out_name=final_fc_name,
    geometry_type="POLYGON",
    spatial_reference=target_sr
)

arcpy.management.AddField(final_fc, "SRC", "TEXT", field_length=10)
arcpy.management.AddField(final_fc, "SRC_PATH", "TEXT", field_length=255)

# ============================================================
# PROCESS EACH DATASET
# Strategy:
# 1. select features touching boundary
# 2. copy selected
# 3. repair geometry
# 4. project if needed
# 5. pairwise clip to boundary
# 6. pairwise erase lake overlap
# 7. append to final
# ============================================================
msg("\nProcessing wetlands datasets...")

for src_name, in_fc in wetland_inputs.items():
    msg("\n" + "=" * 70)
    msg(f"Working on: {src_name}")
    msg(f"Input: {in_fc}")

    lyr = f"lyr_{src_name}"
    sel_fc = os.path.join(work_gdb, f"{src_name}_sel")
    prj_fc = os.path.join(work_gdb, f"{src_name}_prj")
    clp_fc = os.path.join(work_gdb, f"{src_name}_clip")
    ers_fc = os.path.join(work_gdb, f"{src_name}_erase")

    # clean old temp
    for tmp in [lyr, sel_fc, prj_fc, clp_fc, ers_fc]:
        safe_delete(tmp)

    try:
        # make feature layer
        arcpy.management.MakeFeatureLayer(in_fc, lyr)

        # select only features touching boundary to reduce processing load
        arcpy.management.SelectLayerByLocation(
            in_layer=lyr,
            overlap_type="INTERSECT",
            select_features=boundary_proj,
            selection_type="NEW_SELECTION"
        )

        nsel = get_count(lyr)
        msg(f"Selected features intersecting boundary: {nsel:,}")

        if nsel == 0:
            msg(f"No features intersect boundary for {src_name}; skipping.")
            arcpy.management.Delete(lyr)
            continue

        # copy selected subset to gdb
        arcpy.management.CopyFeatures(lyr, sel_fc)
        arcpy.management.Delete(lyr)

        # repair geometry
        arcpy.management.RepairGeometry(sel_fc, "DELETE_NULL")

        # project to target CRS
        project_if_needed(sel_fc, prj_fc, target_sr)
        arcpy.management.RepairGeometry(prj_fc, "DELETE_NULL")

        # clip to boundary
        arcpy.analysis.PairwiseClip(prj_fc, boundary_proj, clp_fc)
        nclip = get_count(clp_fc)
        msg(f"Features after clip: {nclip:,}")

        if nclip == 0:
            msg(f"No features remain after clip for {src_name}; skipping.")
            continue

        arcpy.management.RepairGeometry(clp_fc, "DELETE_NULL")

        # erase the Great Lakes footprint from wetlands
        arcpy.analysis.PairwiseErase(clp_fc, lakes_proj, ers_fc)
        ners = get_count(ers_fc)
        msg(f"Features after lake erase: {ners:,}")

        if ners == 0:
            msg(f"No features remain after lake erase for {src_name}; skipping.")
            continue

        arcpy.management.RepairGeometry(ers_fc, "DELETE_NULL")

        # add source fields
        if not has_field(ers_fc, "SRC"):
            arcpy.management.AddField(ers_fc, "SRC", "TEXT", field_length=10)
        if not has_field(ers_fc, "SRC_PATH"):
            arcpy.management.AddField(ers_fc, "SRC_PATH", "TEXT", field_length=255)

        arcpy.management.CalculateField(ers_fc, "SRC", f"'{src_name}'", "PYTHON3")
        safe_path = in_fc.replace("\\", "/")
        arcpy.management.CalculateField(ers_fc, "SRC_PATH", f"'{safe_path}'", "PYTHON3")

        # append to final
        arcpy.management.Append(
            inputs=ers_fc,
            target=final_fc,
            schema_type="NO_TEST"
        )

        nfinal = get_count(final_fc)
        msg(f"Running total in final output: {nfinal:,}")

    except Exception as e:
        msg(f"ERROR while processing {src_name}: {e}")
        raise

    finally:
        # optional cleanup of temps to keep gdb small
        for tmp in [lyr, sel_fc, prj_fc, clp_fc, ers_fc]:
            safe_delete(tmp)

# ============================================================
# OPTIONAL: DISSOLVE
# Only do this if you want one continuous wetlands layer.
# For very large data, this can take time.
# ============================================================
if do_dissolve:
    msg("\nDissolving final wetlands layer...")
    dissolved_fc = os.path.join(work_gdb, f"{final_fc_name}_diss")
    safe_delete(dissolved_fc)

    arcpy.management.Dissolve(
        in_features=final_fc,
        out_feature_class=dissolved_fc,
        multi_part="MULTI_PART"
    )

    arcpy.management.RepairGeometry(dissolved_fc, "DELETE_NULL")
    final_fc = dissolved_fc
    msg(f"Dissolved output: {final_fc}")

# ============================================================
# FINAL REPAIR + OPTIONAL SHAPEFILE EXPORT
# ============================================================
msg("\nFinal cleanup...")
arcpy.management.RepairGeometry(final_fc, "DELETE_NULL")
final_count = get_count(final_fc)
msg(f"Final wetland feature count: {final_count:,}")

# build spatial index
try:
    arcpy.management.AddSpatialIndex(final_fc)
except Exception:
    pass

if export_shp:
    out_shp = os.path.join(Wetlands_GL, f"{os.path.basename(final_fc_name)}.shp")
    safe_delete(out_shp)
    msg("\nExporting shapefile copy...")
    arcpy.management.CopyFeatures(final_fc, out_shp)
    msg(f"Shapefile exported to: {out_shp}")

msg("\n✅ DONE")
msg(f"Authoritative output feature class:\n{final_fc}")

### Add Elevation to the wetlands from DEM 
We can use zonal statistics methods from arcgis to extract the elevations from the DEM. Following code estimate the min/max/median/average elevation withis the wetlands polygons from the DEM.

In [ ]:
import os
import arcpy
from arcpy.sa import ZonalStatisticsAsTable

arcpy.env.overwriteOutput = True
arcpy.CheckOutExtension("Spatial")

Wetlands_GL = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\Wetlands"
final_gdb_name = "wetlands_processing.gdb"
final_fc_name  = "Wetlands_GL_NoLakes"
DEM = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\DEM\DEM_merged_3174_30m.tif"

wetlands_fc = os.path.join(Wetlands_GL, final_gdb_name, final_fc_name)
out_table   = os.path.join(Wetlands_GL, final_gdb_name, "wetland_dem_stats")
zone_field  = "WET_ID"

print("Wetlands FC:", wetlands_fc, "| Exists:", arcpy.Exists(wetlands_fc))
print("DEM:", DEM, "| Exists:", arcpy.Exists(DEM))

if not arcpy.Exists(wetlands_fc):
    raise RuntimeError(f"Wetland feature class not found: {wetlands_fc}")
if not arcpy.Exists(DEM):
    raise RuntimeError(f"DEM not found: {DEM}")

# refresh extent in case it is stale
arcpy.management.RecalculateFeatureClassExtent(wetlands_fc)

wet_desc = arcpy.Describe(wetlands_fc)
dem_desc = arcpy.Describe(DEM)

arcpy.env.snapRaster = DEM
arcpy.env.cellSize = DEM
arcpy.env.outputCoordinateSystem = dem_desc.spatialReference
arcpy.env.extent = wet_desc.extent   # safer than arcpy.env.extent = wetlands_fc

existing_fields = [f.name for f in arcpy.ListFields(wetlands_fc)]
if zone_field not in existing_fields:
    arcpy.management.AddField(wetlands_fc, zone_field, "LONG")
    arcpy.management.CalculateField(wetlands_fc, zone_field, "!OBJECTID!", "PYTHON3")

if arcpy.Exists(out_table):
    arcpy.management.Delete(out_table)

ZonalStatisticsAsTable(
    in_zone_data=wetlands_fc,
    zone_field=zone_field,
    in_value_raster=DEM,
    out_table=out_table,
    ignore_nodata="DATA",
    statistics_type="ALL"
)

print("Done:", out_table)

#### merge the table with shapefile 

In [ ]:
import os
import arcpy

arcpy.env.overwriteOutput = True

# ============================================================
# INPUTS
# ============================================================
Wetlands_GL = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\Wetlands"
final_gdb_name = "wetlands_processing.gdb"
final_fc_name  = "Wetlands_GL_NoLakes"

wetlands_fc = os.path.join(Wetlands_GL, final_gdb_name, final_fc_name)
stats_table = os.path.join(Wetlands_GL, final_gdb_name, "wetland_dem_stats")

zone_field = "WET_ID"   # must exist in both FC and stats table

print("Wetlands FC:", wetlands_fc, "| Exists:", arcpy.Exists(wetlands_fc))
print("Stats table:", stats_table, "| Exists:", arcpy.Exists(stats_table))

if not arcpy.Exists(wetlands_fc):
    raise RuntimeError(f"Wetlands feature class not found:\n{wetlands_fc}")

if not arcpy.Exists(stats_table):
    raise RuntimeError(f"Stats table not found:\n{stats_table}")

# ============================================================
# CHECK FIELDS
# ============================================================
wet_fields = [f.name for f in arcpy.ListFields(wetlands_fc)]
tbl_fields = [f.name for f in arcpy.ListFields(stats_table)]

print("\nWetland FC fields:")
print(wet_fields)

print("\nStats table fields:")
print(tbl_fields)

if zone_field not in wet_fields:
    raise RuntimeError(f"{zone_field} not found in wetlands FC")

if zone_field not in tbl_fields:
    raise RuntimeError(f"{zone_field} not found in stats table")

# ============================================================
# CHOOSE STAT FIELDS TO JOIN
# ============================================================
# ArcGIS zonal stats usually creates names like MIN, MAX, MEAN, MEDIAN
# but this keeps it flexible if names vary slightly.
tbl_upper = {f.name.upper(): f.name for f in arcpy.ListFields(stats_table)}

min_field    = tbl_upper.get("MIN") or tbl_upper.get("MINIMUM")
max_field    = tbl_upper.get("MAX") or tbl_upper.get("MAXIMUM")
mean_field   = tbl_upper.get("MEAN")
median_field = tbl_upper.get("MEDIAN")

join_fields = [f for f in [min_field, max_field, mean_field, median_field] if f]

print("\nFields that will be joined:")
print(join_fields)

if not join_fields:
    raise RuntimeError("No DEM stats fields found to join.")

# ============================================================
# DELETE OLD OUTPUT FIELDS IF THEY ALREADY EXIST
# ============================================================
# We create our own clean permanent fields after the join.
target_fields = ["DEM_MIN", "DEM_MAX", "DEM_MEAN", "DEM_MED", "DRN_ELEV"]
existing = [f.name for f in arcpy.ListFields(wetlands_fc)]

to_delete = [f for f in target_fields if f in existing]
if to_delete:
    print("\nDeleting old fields:", to_delete)
    arcpy.management.DeleteField(wetlands_fc, to_delete)

# ============================================================
# JOIN THE TABLE FIELDS INTO THE POLYGON FC
# ============================================================
print("\nJoining DEM stats table to wetlands FC...")
arcpy.management.JoinField(
    in_data=wetlands_fc,
    in_field=zone_field,
    join_table=stats_table,
    join_field=zone_field,
    fields=join_fields
)

# ============================================================
# ADD CLEAN FIELDS
# ============================================================
existing = [f.name for f in arcpy.ListFields(wetlands_fc)]
for fld in target_fields:
    if fld not in existing:
        arcpy.management.AddField(wetlands_fc, fld, "DOUBLE")

# ============================================================
# COPY JOINED VALUES TO CLEAN FIELDS
# ============================================================
def calc_if_exists(target_field, source_field):
    current_fields = [f.name for f in arcpy.ListFields(wetlands_fc)]
    if source_field and source_field in current_fields:
        print(f"Calculating {target_field} from {source_field}")
        arcpy.management.CalculateField(
            wetlands_fc,
            target_field,
            f"!{source_field}!",
            "PYTHON3"
        )
    else:
        print(f"Skipping {target_field}; source field not found.")

calc_if_exists("DEM_MIN",  min_field)
calc_if_exists("DEM_MAX",  max_field)
calc_if_exists("DEM_MEAN", mean_field)
calc_if_exists("DEM_MED",  median_field)

# first-pass drain elevation
if "DEM_MED" in [f.name for f in arcpy.ListFields(wetlands_fc)]:
    arcpy.management.CalculateField(
        wetlands_fc,
        "DRN_ELEV",
        "!DEM_MED!",
        "PYTHON3"
    )

print("\nDone. Added fields to polygon FC:")
print(["DEM_MIN", "DEM_MAX", "DEM_MEAN", "DEM_MED", "DRN_ELEV"])

## 7.3 Merging Drains and find the elevation 

#### 
1-Merge InputStream and InputWetland
2- Covert them to raster name drain.tif save in Outpath
3- use arcpy mask and overlat the Drain with DEM with there DEM intersect with DRain appyly 1 and if there is not any intersect set arcpy.nan them myltiply the raster to DEM 


In [ ]:
import os
import arcpy
import numpy as np
from arcpy.sa import *

arcpy.env.overwriteOutput = True
arcpy.CheckOutExtension("Spatial")

# ============================================================
# INPUTS
# ============================================================
InputStream = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\NHD\streams_merged_clip_to_bdry.shp"

# Put the real readable wetland feature class here if you have it.
# Example:
# InputWetland = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\Wetlands\wetlands_processing.gdb\Wetlands_GL_NoLakes_clean"
InputWetland = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\Wetlands\wetlands_processing.gdb\Wetlands_GL_NoLakes"

DEM = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\DEM\DEM_merged_3174_30m.tif"
Outpath = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\Drains"

# ============================================================
# OUTPUTS
# ============================================================
temp_gdb = os.path.join(Outpath, "drain_work.gdb")

stream_fc   = os.path.join(temp_gdb, "stream_fc")
wetland_fc  = os.path.join(temp_gdb, "wetland_fc")

stream_ras  = os.path.join(temp_gdb, "stream_ras")
wetland_ras = os.path.join(temp_gdb, "wetland_ras")
drain_mask_ras = os.path.join(temp_gdb, "drain_mask_ras")
drain_elev_ras = os.path.join(temp_gdb, "drain_elev_ras")

drain_tif      = os.path.join(Outpath, "drain.tif")
drain_elev_tif = os.path.join(Outpath, "drain_elevation.tif")

# ============================================================
# CHECK INPUTS
# ============================================================
print("InputStream exists:", arcpy.Exists(InputStream))
print("InputWetland exists:", arcpy.Exists(InputWetland))
print("DEM exists:", arcpy.Exists(DEM))

if not arcpy.Exists(InputStream):
    raise ValueError(f"Stream input not found:\n{InputStream}")
if not arcpy.Exists(DEM):
    raise ValueError(f"DEM not found:\n{DEM}")

use_wetland = arcpy.Exists(InputWetland)
print("Use wetland input:", use_wetland)

# ============================================================
# CREATE OUTPUT FOLDER + GDB
# ============================================================
if not os.path.isdir(Outpath):
    os.makedirs(Outpath)

if not arcpy.Exists(temp_gdb):
    arcpy.management.CreateFileGDB(Outpath, "drain_work.gdb")
    print("Created:", temp_gdb)

# ============================================================
# ENVIRONMENT: MATCH DEM EXACTLY
# ============================================================
arcpy.env.snapRaster = DEM
arcpy.env.cellSize = DEM
arcpy.env.extent = DEM
arcpy.env.outputCoordinateSystem = arcpy.Describe(DEM).spatialReference

print("Snap raster:", arcpy.env.snapRaster)
print("Cell size:", arcpy.env.cellSize)
print("Extent:", arcpy.env.extent)

# ============================================================
# CLEAN OLD OUTPUTS
# ============================================================
for item in [
    stream_fc, wetland_fc,
    stream_ras, wetland_ras, drain_mask_ras, drain_elev_ras,
    drain_tif, drain_elev_tif
]:
    if arcpy.Exists(item):
        arcpy.management.Delete(item)

# ============================================================
# COPY INPUTS TO GDB
# ============================================================
arcpy.management.CopyFeatures(InputStream, stream_fc)
print("Copied stream input to:", stream_fc)

if use_wetland:
    arcpy.management.CopyFeatures(InputWetland, wetland_fc)
    print("Copied wetland input to:", wetland_fc)
else:
    print("Wetland input not found/readable. Proceeding with streams only.")

# ============================================================
# ADD CONSTANT BURN FIELD
# ============================================================
burn_field = "BURN"

def ensure_burn_field(fc, burn_field="BURN"):
    fields = [f.name for f in arcpy.ListFields(fc)]
    if burn_field not in fields:
        arcpy.management.AddField(fc, burn_field, "SHORT")
    arcpy.management.CalculateField(fc, burn_field, 1, "PYTHON3")

ensure_burn_field(stream_fc, burn_field)

if use_wetland:
    ensure_burn_field(wetland_fc, burn_field)

print("Added burn field(s).")

# ============================================================
# 1) CONVERT STREAMS TO RASTER
# ============================================================
arcpy.conversion.PolylineToRaster(
    in_features=stream_fc,
    value_field=burn_field,
    out_rasterdataset=stream_ras,
    cell_assignment="MAXIMUM_LENGTH",
    priority_field="NONE",
    cellsize=DEM
)
print("Created stream raster:", stream_ras)

# ============================================================
# 2) CONVERT WETLANDS TO RASTER (OPTIONAL)
# ============================================================
if use_wetland:
    arcpy.conversion.PolygonToRaster(
        in_features=wetland_fc,
        value_field=burn_field,
        out_rasterdataset=wetland_ras,
        cell_assignment="CELL_CENTER",
        priority_field="NONE",
        cellsize=DEM
    )
    print("Created wetland raster:", wetland_ras)

# ============================================================
# 3) COMBINE TO ONE DRAIN MASK
# drain = 1 where stream OR wetland exists
# ============================================================
streamR = Raster(stream_ras)

if use_wetland:
    wetlandR = Raster(wetland_ras)
    drain_mask = Con((~IsNull(streamR)) | (~IsNull(wetlandR)), 1)
else:
    drain_mask = Con(~IsNull(streamR), 1)

drain_mask.save(drain_mask_ras)
print("Created combined drain mask raster:", drain_mask_ras)

# export to tif
arcpy.management.CopyRaster(drain_mask_ras, drain_tif)
print("Exported:", drain_tif)

# ============================================================
# 4) OVERLAY DRAIN MASK WITH DEM
# Keep DEM elevation only where drain exists
# ============================================================
demR = Raster(DEM)
drain_elev = Con(drain_mask == 1, demR)
drain_elev.save(drain_elev_ras)
print("Created drain elevation raster:", drain_elev_ras)

# export to tif
arcpy.management.CopyRaster(drain_elev_ras, drain_elev_tif)
print("Exported:", drain_elev_tif)

# ============================================================
# 5) NUMPY ARRAYS
# arrayDrnElev = nan where no drain, else DEM elevation
# ============================================================
# arrayDrainMask = arcpy.RasterToNumPyArray(drain_mask_ras, nodata_to_value=0).astype(float)
# arrayDEM = arcpy.RasterToNumPyArray(DEM, nodata_to_value=np.nan).astype(float)

# arrayDrnElev = np.where(arrayDrainMask == 0, np.nan, 1.0) * arrayDEM

# print("arrayDrnElev shape:", arrayDrnElev.shape)
# print("Drain cells in array:", int(np.sum(~np.isnan(arrayDrnElev))))
# print("Drain elevation min:", float(np.nanmin(arrayDrnElev)))
# print("Drain elevation max:", float(np.nanmax(arrayDrnElev)))

# ============================================================
# 6) QUICK CHECKS
# ============================================================
print("Drain mask min:",
      arcpy.management.GetRasterProperties(drain_mask_ras, "MINIMUM").getOutput(0))
print("Drain mask max:",
      arcpy.management.GetRasterProperties(drain_mask_ras, "MAXIMUM").getOutput(0))

print("Drain elevation min:",
      arcpy.management.GetRasterProperties(drain_elev_ras, "MINIMUM").getOutput(0))
print("Drain elevation max:",
      arcpy.management.GetRasterProperties(drain_elev_ras, "MAXIMUM").getOutput(0))

print("Done.")

# 8. Great Lakes bathymetry and modified model top
Prepare lake bathymetry contours, standardize them, rasterize them, and use them to update the model top.


In [3]:
import arcpy
# Create a lake raster where lakes cells are 1 
Greatlakes =r"D:\Users\abolmaal\Arcgis\GreatLakes\hydro_p_GreatLakes.shp"
lake_ras = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\Lakes\GreatLakes_ras.tif"
DEM= r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\DEM\DEM_extended20kmbdr_1000m.tif"


# arcpy.management.AddField(Greatlakes, "lake_val", "SHORT")
# arcpy.management.CalculateField(Greatlakes, "lake_val", 1)

arcpy.conversion.PolygonToRaster(
    in_features=Greatlakes,
    value_field="lake_val",
    out_rasterdataset=lake_ras,
    cell_assignment="MAXIMUM_AREA",
    priority_field="NONE",
    cellsize=DEM
)

<Result 'D:\\Users\\abolmaal\\modelling\\Modflow\\Prep\\GreatLakes\\model_Layers\\Lakes\\GreatLakes_ras.tif'>

### Clip Lake Superior contours to the lake extent

Clips the Superior contour lines to the Lake Superior polygon so only relevant bathymetry contours remain.


In [ ]:


# clip out_countour to lake superior extent (optional)
arcpy.env.overwriteOutput = True

out_contour = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\GreatLakes_bathymetry\superior_contours.shp"

Sub_Superior = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\Lakes\subbasins\hydro_p_LakeSuperior.shp"

out_clip = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\GreatLakes_bathymetry\out_contour_LakeSuperior.shp"


print("Contour SR:", arcpy.Describe(out_contour).spatialReference.name)
print("Clip SR:", arcpy.Describe(Sub_Superior).spatialReference.name)

arcpy.analysis.Clip(
    in_features=out_contour,
    clip_features=Sub_Superior,
    out_feature_class=out_clip
)

print("Saved:", out_clip)
print("Count:", arcpy.management.GetCount(out_clip)[0])

### Inspect bathymetry attribute fields

Checks which depth/elevation fields exist in each lake bathymetry shapefile so a consistent merge field can be built.


In [ ]:
import arcpy
import os

files = [
    r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\GreatLakes_bathymetry\out_contour_LakeSuperior.shp",
    r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\GreatLakes_bathymetry\Lake_Michigan_Contours.shp",
    r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\GreatLakes_bathymetry\Lake_Huron_Contours.shp",
    r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\GreatLakes_bathymetry\Lake_Erie_Contours.shp",
    r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\GreatLakes_bathymetry\Lake_Ontario_Contours.shp",
]

check_fields = ["DEPTH", "Contour", "DEPTH_SL", "ZVALUE"]

for fc in files:
    print("\n====================================================")
    print(os.path.basename(fc))
    fields = [f.name for f in arcpy.ListFields(fc)]
    print("Fields:", fields)

    for fld in check_fields:
        if fld in fields:
            total = 0
            non_null = 0
            sample = []
            with arcpy.da.SearchCursor(fc, [fld]) as cur:
                for row in cur:
                    total += 1
                    v = row[0]
                    if v is not None:
                        non_null += 1
                        if len(sample) < 5:
                            sample.append(v)
            print(f"  {fld}: non-null {non_null}/{total}, sample={sample}")

### Standardize and merge all lake bathymetry contours

Projects, standardizes attributes, and merges the Great Lakes bathymetry contour layers into a common schema. This is the cleaner, more finalized merge workflow.


In [ ]:
import arcpy
import os

arcpy.env.overwriteOutput = True

# ---------------------------------------------------
# INPUTS
# ---------------------------------------------------
sup_bathy = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\GreatLakes_bathymetry\out_contour_LakeSuperior.shp"
Michigan_bathy = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\GreatLakes_bathymetry\Lake_Michigan_Contours.shp"
Huron_bathy = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\GreatLakes_bathymetry\Lake_Huron_Contours.shp"
Erie_bathy = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\GreatLakes_bathymetry\Lake_Erie_Contours.shp"
Ontario_bathy = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\GreatLakes_bathymetry\Lake_Ontario_Contours.shp"

out_merged = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\GreatLakes_bathymetry\GreatLakes_bathymetry_final.shp"

inputs = {
    "Superior": sup_bathy,
    "Michigan": Michigan_bathy,
    "Huron": Huron_bathy,
    "Erie": Erie_bathy,
    "Ontario": Ontario_bathy,
}

# ---------------------------------------------------
# WORKSPACE
# ---------------------------------------------------
work_dir = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\GreatLakes_bathymetry"
temp_gdb = os.path.join(work_dir, "temp_bathy_merge.gdb")

if arcpy.Exists(temp_gdb):
    arcpy.management.Delete(temp_gdb)

arcpy.management.CreateFileGDB(work_dir, "temp_bathy_merge.gdb")
print("Created temp GDB:", temp_gdb)

target_sr = arcpy.Describe(sup_bathy).spatialReference
print("Target CRS:", target_sr.name, "|", target_sr.factoryCode)

standardized_inputs = []

# ---------------------------------------------------
# HELPERS
# ---------------------------------------------------
def field_exists(fc, name):
    return name in [f.name for f in arcpy.ListFields(fc)]

def add_field_if_needed(fc, name, field_type, field_length=None):
    if not field_exists(fc, name):
        if field_length is not None:
            arcpy.management.AddField(fc, name, field_type, field_length=field_length)
        else:
            arcpy.management.AddField(fc, name, field_type)

def populate_depth_field(fc, lake_name):
    """
    Populate DEPTH_M using lake-specific logic:
      Superior -> abs(Contour)
      Others   -> DEPTH
    """
    fields = [f.name for f in arcpy.ListFields(fc)]

    add_field_if_needed(fc, "DEPTH_M", "DOUBLE")
    add_field_if_needed(fc, "LAKE_NAME", "TEXT", field_length=20)

    if lake_name == "Superior":
        if "Contour" not in fields:
            raise RuntimeError(f"'Contour' field not found in {fc}")

        cursor_fields = ["Contour", "DEPTH_M", "LAKE_NAME"]
        with arcpy.da.UpdateCursor(fc, cursor_fields) as cur:
            for row in cur:
                contour_val = row[0]
                row[1] = abs(float(contour_val)) if contour_val is not None else None
                row[2] = lake_name
                cur.updateRow(row)

    else:
        if "DEPTH" not in fields:
            raise RuntimeError(f"'DEPTH' field not found in {fc}")

        cursor_fields = ["DEPTH", "DEPTH_M", "LAKE_NAME"]
        with arcpy.da.UpdateCursor(fc, cursor_fields) as cur:
            for row in cur:
                depth_val = row[0]
                row[1] = float(depth_val) if depth_val is not None else None
                row[2] = lake_name
                cur.updateRow(row)

    # summary
    total = 0
    non_null = 0
    sample = []
    with arcpy.da.SearchCursor(fc, ["DEPTH_M"]) as cur:
        for row in cur:
            total += 1
            if row[0] is not None:
                non_null += 1
                if len(sample) < 5:
                    sample.append(row[0])

    print(f"  DEPTH_M populated: {non_null}/{total}, sample={sample}")


# ---------------------------------------------------
# COPY -> REPAIR -> PROJECT -> STANDARDIZE
# ---------------------------------------------------
for lake_name, fc in inputs.items():
    print("\n===================================================")
    print("Processing:", lake_name)
    print("Input:", fc)

    if not arcpy.Exists(fc):
        raise FileNotFoundError(fc)

    sr = arcpy.Describe(fc).spatialReference
    print("  Original CRS:", sr.name, "|", sr.factoryCode)
    print("  Original count:", arcpy.management.GetCount(fc)[0])

    short = lake_name[:3].lower()

    # copy
    fc_copy = os.path.join(temp_gdb, f"{short}_copy")
    if arcpy.Exists(fc_copy):
        arcpy.management.Delete(fc_copy)
    arcpy.management.CopyFeatures(fc, fc_copy)

    # repair
    try:
        arcpy.management.RepairGeometry(fc_copy)
    except Exception as e:
        print("  RepairGeometry warning:", e)

    print("  Count after copy/repair:", arcpy.management.GetCount(fc_copy)[0])

    # project
    same_crs = (
        sr.factoryCode == target_sr.factoryCode
        and sr.name == target_sr.name
    )

    if same_crs:
        fc_proj = fc_copy
        print("  Already in target CRS.")
    else:
        fc_proj = os.path.join(temp_gdb, f"{short}_proj")
        if arcpy.Exists(fc_proj):
            arcpy.management.Delete(fc_proj)

        tx = arcpy.ListTransformations(sr, target_sr)
        transform = tx[0] if tx else None

        if transform:
            print("  Using transformation:", transform)
            arcpy.management.Project(fc_copy, fc_proj, target_sr, transform)
        else:
            print("  No transformation returned; projecting without explicit transform.")
            arcpy.management.Project(fc_copy, fc_proj, target_sr)

        print("  Count after project:", arcpy.management.GetCount(fc_proj)[0])

    # standardize attributes
    populate_depth_field(fc_proj, lake_name)

    standardized_inputs.append(fc_proj)

# ---------------------------------------------------
# MERGE
# ---------------------------------------------------
if arcpy.Exists(out_merged):
    arcpy.management.Delete(out_merged)

arcpy.management.Merge(standardized_inputs, out_merged)

print("\nMerged output saved to:")
print(out_merged)
print("Merged count:", arcpy.management.GetCount(out_merged)[0])

# ---------------------------------------------------
# FINAL CHECK
# ---------------------------------------------------
print("\nFinal fields:")
for f in arcpy.ListFields(out_merged):
    print(" ", f.name, "|", f.type)

# DEPTH_M summary
total = 0
non_null = 0
sample = []
with arcpy.da.SearchCursor(out_merged, ["LAKE_NAME", "DEPTH_M"]) as cur:
    for row in cur:
        total += 1
        if row[1] is not None:
            non_null += 1
            if len(sample) < 10:
                sample.append(row)

print("\nDEPTH_M non-null:", non_null, "/", total)
print("Sample rows:", sample)

### Rasterize merged lake bathymetry contours

Prepares the merged bathymetry contours and converts them to a raster that can be combined with the model-top DEM.


In [ ]:
import arcpy
import os

arcpy.env.overwriteOutput = True

# ---------------------------------------------------
# PATHS
# ---------------------------------------------------
work_dir = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\GreatLakes_bathymetry"
temp_gdb = os.path.join(work_dir, "temp_bathy_raster.gdb")

sup_fc = os.path.join(temp_gdb, "sup_copy")   # already target CRS
mic_fc = os.path.join(temp_gdb, "mic_proj")
hur_fc = os.path.join(temp_gdb, "hur_proj")
eri_fc = os.path.join(temp_gdb, "eri_proj")
ont_fc = os.path.join(temp_gdb, "ont_proj")

prepared_inputs = [sup_fc, mic_fc, hur_fc, eri_fc, ont_fc]

out_merged_fc = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\GreatLakes_bathymetry\GreatLakes_bathymetry_merged.shp"
out_raster = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\GreatLakes_bathymetry\GreatLakes_bathymetry_contours.tif"

cellsize = 1000  # change if needed


# ---------------------------------------------------
# HELPERS
# ---------------------------------------------------
def field_exists(fc, name):
    return name in [f.name for f in arcpy.ListFields(fc)]

def add_field_if_needed(fc, name, field_type, field_length=None):
    if not field_exists(fc, name):
        if field_length is not None:
            arcpy.management.AddField(fc, name, field_type, field_length=field_length)
        else:
            arcpy.management.AddField(fc, name, field_type)

def populate_bathy_z(fc, lake_name):
    """
    Keep values NEGATIVE.
    Superior: use Contour exactly as-is.
    Others:   use ZVALUE if present; otherwise use -DEPTH.
    """
    fields = [f.name for f in arcpy.ListFields(fc)]

    add_field_if_needed(fc, "BATHY_Z", "DOUBLE")
    add_field_if_needed(fc, "LAKE_NAME", "TEXT", field_length=20)

    if lake_name == "Superior":
        if "Contour" not in fields:
            raise RuntimeError(f"Contour field not found in {fc}")

        with arcpy.da.UpdateCursor(fc, ["Contour", "BATHY_Z", "LAKE_NAME"]) as cur:
            for row in cur:
                row[1] = float(row[0]) if row[0] is not None else None
                row[2] = lake_name
                cur.updateRow(row)

    else:
        if "ZVALUE" in fields:
            with arcpy.da.UpdateCursor(fc, ["ZVALUE", "BATHY_Z", "LAKE_NAME"]) as cur:
                for row in cur:
                    row[1] = float(row[0]) if row[0] is not None else None
                    row[2] = lake_name
                    cur.updateRow(row)

        elif "DEPTH" in fields:
            with arcpy.da.UpdateCursor(fc, ["DEPTH", "BATHY_Z", "LAKE_NAME"]) as cur:
                for row in cur:
                    row[1] = -float(row[0]) if row[0] is not None else None
                    row[2] = lake_name
                    cur.updateRow(row)

        else:
            raise RuntimeError(f"No ZVALUE or DEPTH field found in {fc}")

    # quick check
    total = 0
    non_null = 0
    sample = []
    with arcpy.da.SearchCursor(fc, ["BATHY_Z"]) as cur:
        for row in cur:
            total += 1
            if row[0] is not None:
                non_null += 1
                if len(sample) < 5:
                    sample.append(row[0])

    print(f"{lake_name}: BATHY_Z non-null {non_null}/{total}, sample={sample}")


# ---------------------------------------------------
# POPULATE BATHY_Z
# ---------------------------------------------------
populate_bathy_z(sup_fc, "Superior")
populate_bathy_z(mic_fc, "Michigan")
populate_bathy_z(hur_fc, "Huron")
populate_bathy_z(eri_fc, "Erie")
populate_bathy_z(ont_fc, "Ontario")


# ---------------------------------------------------
# MERGE
# ---------------------------------------------------
if arcpy.Exists(out_merged_fc):
    arcpy.management.Delete(out_merged_fc)

arcpy.management.Merge(prepared_inputs, out_merged_fc)

print("\nMerged feature class:")
print(out_merged_fc)
print("Merged count:", arcpy.management.GetCount(out_merged_fc)[0])


# ---------------------------------------------------
# CONVERT MERGED CONTOURS TO RASTER
# ---------------------------------------------------
if arcpy.Exists(out_raster):
    arcpy.management.Delete(out_raster)

arcpy.conversion.PolylineToRaster(
    in_features=out_merged_fc,
    value_field="BATHY_Z",
    out_rasterdataset=out_raster,
    cell_assignment="MAXIMUM_LENGTH",
    priority_field="NONE",
    cellsize=cellsize
)

print("\nRaster saved to:")
print(out_raster)


# ---------------------------------------------------
# FINAL CHECK
# ---------------------------------------------------
print("\nMerged fields:")
for f in arcpy.ListFields(out_merged_fc):
    print(f"  {f.name} | {f.type}")

print("\nSample merged values:")
with arcpy.da.SearchCursor(out_merged_fc, ["LAKE_NAME", "BATHY_Z"]) as cur:
    for i, row in enumerate(cur):
        print(row)
        if i == 9:
            break

### Modify model top with lake bathymetry

Uses the lake bathymetry raster to lower the model top where lakes have negative bathymetric values, creating an updated top surface for the groundwater model.
Modify the Model top elevation by subtracting the negative cells of Lake_bathymetry from the model_top

In [ ]:
import os
import arcpy
from arcpy.sa import Raster, Con, IsNull

# ---------------------------------------------------
# SETTINGS
# ---------------------------------------------------
arcpy.env.overwriteOutput = True
arcpy.CheckOutExtension("Spatial")

# ---------------------------------------------------
# INPUTS
# ---------------------------------------------------
model_top = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\DEM\DEM_merged_3174_30m.tif"
lake_bathy = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\GreatLakes_bathymetry\GreatLakes_bathymetry_contours.tif"
boundary_shp = r"S:\Projects\Active\GLB_LHM\LHM_inputs\Boundary\extendedBdry_jan26_adk.shp"

# Output rasters
full_modified = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\Top\top_Lakebathy_full_30m.tif"
final_modified = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\Top\top_Lakebathy_30m.tif"

# Scratch workspace
scratch_folder = r"D:\Users\abolmaal\modelling\Modflow\scratch"
os.makedirs(scratch_folder, exist_ok=True)

scratch_gdb = os.path.join(scratch_folder, "scratch.gdb")
if not arcpy.Exists(scratch_gdb):
    arcpy.management.CreateFileGDB(scratch_folder, "scratch.gdb")

# Temporary datasets
boundary_use = os.path.join(scratch_gdb, "boundary_use")
bathy_proj   = os.path.join(scratch_gdb, "bathy_proj")
bathy_30m    = os.path.join(scratch_gdb, "bathy_30m")
diff_raster  = os.path.join(scratch_gdb, "top_diff")

# ---------------------------------------------------
# HELPER FUNCTIONS
# ---------------------------------------------------
def safe_delete(*paths):
    """Delete datasets if they exist."""
    for p in paths:
        if arcpy.Exists(p):
            arcpy.management.Delete(p)

def crs_match(sr1, sr2):
    """Compare two spatial references by factory code first, then WKT."""
    if sr1.factoryCode and sr1.factoryCode != 0 and sr1.factoryCode == sr2.factoryCode:
        return True
    return sr1.exportToString() == sr2.exportToString()

def finalize_raster(path):
    """Calculate statistics and build pyramids for a finished raster."""
    arcpy.management.CalculateStatistics(path)
    arcpy.management.BuildPyramids(path)

# ---------------------------------------------------
# DESCRIBE INPUTS
# ---------------------------------------------------
top_desc = arcpy.Describe(model_top)
top_sr   = top_desc.spatialReference
top_ext  = top_desc.extent
top_r    = Raster(model_top)
cellsize = float(top_r.meanCellWidth)

bathy_desc = arcpy.Describe(lake_bathy)
bathy_sr   = bathy_desc.spatialReference

bnd_desc = arcpy.Describe(boundary_shp)
bnd_sr   = bnd_desc.spatialReference

print("MODEL TOP CRS:", top_sr.name, top_sr.factoryCode)
print("BATHY CRS    :", bathy_sr.name, bathy_sr.factoryCode)
print("BOUNDARY CRS :", bnd_sr.name, bnd_sr.factoryCode)
print("Model top cell size:", cellsize)

# ---------------------------------------------------
# 1) PREPARE BOUNDARY
# ---------------------------------------------------
if bnd_sr.name == "Unknown":
    raise ValueError("Boundary shapefile has Unknown coordinate system. Define projection first.")

safe_delete(boundary_use)

if not crs_match(top_sr, bnd_sr):
    arcpy.management.Project(boundary_shp, boundary_use, top_sr)
    print("Projected boundary to model_top CRS.")
else:
    arcpy.management.CopyFeatures(boundary_shp, boundary_use)
    print("Boundary already matches model_top CRS.")

# ---------------------------------------------------
# 2) PREPARE BATHYMETRY
# ---------------------------------------------------
if bathy_sr.name == "Unknown":
    raise ValueError("Bathymetry raster has Unknown coordinate system. Define projection first.")

safe_delete(bathy_proj)

if not crs_match(top_sr, bathy_sr):
    arcpy.management.ProjectRaster(
        in_raster=lake_bathy,
        out_raster=bathy_proj,
        out_coor_system=top_sr,
        resampling_type="BILINEAR",
        cell_size=cellsize
    )
    bathy_use = bathy_proj
    print("Projected bathymetry to model_top CRS.")
else:
    bathy_use = lake_bathy
    print("Bathymetry already matches model_top CRS.")

# ---------------------------------------------------
# 3) SET ENVIRONMENT TO THE MODEL_TOP GRID
# ---------------------------------------------------
arcpy.env.snapRaster             = model_top
arcpy.env.cellSize               = model_top
arcpy.env.outputCoordinateSystem = top_sr
arcpy.env.compression            = "LZW"

# ---------------------------------------------------
# 4) RESAMPLE BATHYMETRY TO 30 m
# ---------------------------------------------------
safe_delete(bathy_30m)

arcpy.management.Resample(
    in_raster=bathy_use,
    out_raster=bathy_30m,
    cell_size=f"{cellsize} {cellsize}",
    resampling_type="BILINEAR"
)
print("Created 30 m bathymetry raster.")

# ---------------------------------------------------
# 5) BUILD NEGATIVE BATHYMETRY OFFSET
# NoData cells (outside lake coverage) become 0
# Positive or zero bathy cells become 0
# Negative bathy cells are kept as-is
# ---------------------------------------------------
bathy_r   = Raster(bathy_30m)
neg_bathy = Con(IsNull(bathy_r), 0, Con(bathy_r < 0, bathy_r, 0))

# ---------------------------------------------------
# 6) SAVE FULL-EXTENT MODIFIED RASTER
# DEM is lowered where bathy is negative; unchanged elsewhere
# ---------------------------------------------------
arcpy.env.extent = model_top
arcpy.env.mask   = None

safe_delete(full_modified)
(top_r + neg_bathy).save(full_modified)
finalize_raster(full_modified)
print("Saved full modified raster:", full_modified)

# ---------------------------------------------------
# 7) SAVE BOUNDARY-CLIPPED FINAL RASTER
# Use arcpy.env.mask to clip implicitly — no extra Clip step
# ---------------------------------------------------
arcpy.env.extent = arcpy.Describe(boundary_use).extent
arcpy.env.mask   = boundary_use

safe_delete(final_modified)
(top_r + neg_bathy).save(final_modified)
finalize_raster(final_modified)
print("Saved final clipped raster:", final_modified)

# Reset mask and extent
arcpy.env.mask   = None
arcpy.env.extent = model_top

# ---------------------------------------------------
# 8) QA DIFFERENCE RASTER
# Shows where the DEM was lowered (values will be <= 0)
# ---------------------------------------------------
safe_delete(diff_raster)
(Raster(full_modified) - top_r).save(diff_raster)
print("Saved QA difference raster:", diff_raster)

# ---------------------------------------------------
# 9) PRINT CHECKS
# ---------------------------------------------------
for path in [model_top, bathy_30m, full_modified, final_modified]:
    rr = arcpy.Raster(path)
    e  = arcpy.Describe(path).extent
    print(f"\n{path}")
    print(f"  rows, cols : {rr.height} x {rr.width}")
    print(f"  extent     : {e.XMin:.2f} {e.YMin:.2f} {e.XMax:.2f} {e.YMax:.2f}")

print("\nDone.")


In [ ]:
import arcpy
from arcpy.sa import Raster

model_top = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\DEM\DEM_merged_3174_30m.tif"
modified_top = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\Top\top_Lakebathy_30m.tif"
diff_out = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\Top\top_diff.tif"

diff = Raster(modified_top) - Raster(model_top)
diff.save(diff_out)

arcpy.management.CalculateStatistics(diff_out)
arcpy.management.BuildPyramids(diff_out)

print("Saved difference raster:", diff_out)
print("Min:", arcpy.management.GetRasterProperties(diff_out, "MINIMUM"))
print("Max:", arcpy.management.GetRasterProperties(diff_out, "MAXIMUM"))

## 9. Starting heads from wells and hydrography
This section builds surfaces and control points to estimate initial heads for the MODFLOW model.


### Prepare observation wells and starting heads

Builds well-based and hydrography-informed surfaces used to estimate starting heads for the groundwater model. This section is long because it includes clipping, interpolation, and control-point preparation.

Steps:

1- It extracts static water-table estimates from wells.

2- It computes well water-table elevation using either SWL as depth below ground or as absolute elevation.

3- It runs an initial EBK using wells only.

4- It compares that initial raster to the DEM and finds cells where WT > land surface.

5- It converts those cells to points and assigns those points WT_ELEV = DEM_ELEV.

6- It optionally adds stream/lake control points, also assigned WT_ELEV = DEM_ELEV.

7- It merges the well points plus the surface/hydrology control points.

8- It runs a final EBK to create the starting-head raster.

In [ ]:
import os
import time
import arcpy
from arcpy.sa import Con, IsNull, ExtractMultiValuesToPoints, Raster, Aggregate

# =====================================================================
# LICENSES / GENERAL ENVIRONMENT
# =====================================================================
# This workflow needs:
#   1) Spatial Analyst       -> raster math, ExtractMultiValuesToPoints, Aggregate
#   2) Geostatistical Analyst -> Empirical Bayesian Kriging (EBK)
#
# If either extension is unavailable, the script stops.
# =====================================================================
arcpy.env.overwriteOutput = True

if arcpy.CheckExtension("Spatial") == "Available":
    arcpy.CheckOutExtension("Spatial")
else:
    raise RuntimeError("Spatial Analyst extension is not available.")

if arcpy.CheckExtension("GeoStats") == "Available":
    arcpy.CheckOutExtension("GeoStats")
else:
    raise RuntimeError("Geostatistical Analyst extension is not available.")


# =====================================================================
# INPUTS
# =====================================================================
Well_path   = r"S:\Data\GIS_Data\Derived\Great_Lakes_Basin\Watersheds\Water_Wells\GLB_water_wells.gdb\GLB_all_wells_2025_mi_update"
boundary_p  = r"S:\Projects\Active\GLB_LHM\LHM_inputs\Boundary\extendedBdry_jan26_adk.shp"
dem_tif     = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\ALIGNED_3174\modified_top_1000m.tif"
out_dir     = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\Wells"
os.makedirs(out_dir, exist_ok=True)

# =====================================================================
# OPTIONAL SURFACE HYDROLOGY INPUTS
# =====================================================================
# These are optional extra control features that can be merged with the
# "WT > DEM" points before the final kriging.
#
# Leave these empty if you do not want to use them.
# =====================================================================
hydro_line_fcs = [
    # r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\NHD\streams_3174.shp",
]

hydro_poly_fcs = [
    # r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\Lakes\GreatLakes_polygons.shp",
]

# =====================================================================
# WELL SETTINGS
# =====================================================================
# SWL_FIELD = field holding the static water level information.
#
# SWL_MODE options:
#   "DEPTH_BELOW_GROUND"  -> WT_ELEV = DEM_ELEV - SWL
#   "ELEVATION"           -> WT_ELEV = SWL
#
# Use DEPTH_BELOW_GROUND if SWL is a depth from the ground surface down to
# the water table. Use ELEVATION if the field already stores the water-table
# elevation.
# =====================================================================
SWL_FIELD = "SWL"
SWL_MODE  = "DEPTH_BELOW_GROUND"   # change to "ELEVATION" if needed

# Simple range filter to remove clearly bad values
SWL_MIN = -1000
SWL_MAX = 5000

# =====================================================================
# GRID / INTERPOLATION SETTINGS
# =====================================================================
# FINAL_CELL_SIZE:
#   resolution of your final starting-head raster
#
# INITIAL_CELL_SIZE:
#   coarser resolution for the FIRST EBK only
#   This first interpolation is only used to identify where WT > DEM,
#   so it can be much coarser and much faster than the final raster.
#
# MASK_AGG_FACTOR:
#   optional extra thinning of the WT > DEM mask BEFORE converting to points
#   1 = no aggregation
#   2 = 2x2 block aggregation
#   3 = 3x3 block aggregation
#
# HYDRO_POINT_SPACING_M:
#   spacing for generating points along stream lines
# =====================================================================
FINAL_CELL_SIZE   = 1000
INITIAL_CELL_SIZE = 5000
MASK_AGG_FACTOR   = 2   # set to 1 to disable extra thinning

HYDRO_POINT_SPACING_M = 5000
FILTER_HYDRO_TO_MASK  = True

# =====================================================================
# INITIAL EBK SETTINGS (FAST SCREENING PASS)
# =====================================================================
# These settings are intentionally lighter / faster because the first EBK
# is only used to flag areas where the kriged water table is above land.
#
# Lower values here usually make the first pass much faster.
# =====================================================================
INITIAL_EBK_TRANSFORMATION   = "NONE"
INITIAL_EBK_MAX_LOCAL_POINTS = 40
INITIAL_EBK_OVERLAP_FACTOR   = 0.15
INITIAL_EBK_NUM_SIM          = 30
INITIAL_EBK_SEMIVARIOGRAM    = "POWER"
INITIAL_EBK_RADIUS_M         = 75000
INITIAL_EBK_MAX_NEIGHBORS    = 20
INITIAL_EBK_MIN_NEIGHBORS    = 8
INITIAL_EBK_SECTOR_TYPE      = "ONE_SECTOR"

# =====================================================================
# FINAL EBK SETTINGS (BETTER QUALITY)
# =====================================================================
# These are still lighter than your original very expensive setup, but give
# a stronger final surface than the first pass.
# =====================================================================
FINAL_EBK_TRANSFORMATION   = "NONE"
FINAL_EBK_MAX_LOCAL_POINTS = 60
FINAL_EBK_OVERLAP_FACTOR   = 0.25
FINAL_EBK_NUM_SIM          = 40
FINAL_EBK_SEMIVARIOGRAM    = "POWER"
FINAL_EBK_RADIUS_M         = 100000
FINAL_EBK_MAX_NEIGHBORS    = 24
FINAL_EBK_MIN_NEIGHBORS    = 8
FINAL_EBK_SECTOR_TYPE      = "ONE_SECTOR"

# =====================================================================
# OUTPUTS
# =====================================================================
wrk_gdb = os.path.join(out_dir, "starting_heads_ebk_optimized_work.gdb")
if arcpy.Exists(wrk_gdb):
    arcpy.management.Delete(wrk_gdb)
arcpy.management.CreateFileGDB(out_dir, "starting_heads_ebk_optimized_work.gdb")

# ---- boundary / wells ----
boundary_prj        = os.path.join(wrk_gdb, "boundary_prj")
wells_prj           = os.path.join(wrk_gdb, "wells_prj")
wells_clip          = os.path.join(wrk_gdb, "wells_clip")
wells_valid         = os.path.join(wrk_gdb, "wells_valid")
wells_wt            = os.path.join(wrk_gdb, "wells_wt")
wells_reduced       = os.path.join(wrk_gdb, "wells_reduced")

# ---- surface-control points from WT > DEM ----
surface_pts_raw     = os.path.join(wrk_gdb, "surface_pts_raw")
surface_pts_valid   = os.path.join(wrk_gdb, "surface_pts_valid")
surface_pts_reduced = os.path.join(wrk_gdb, "surface_pts_reduced")

# ---- optional hydro features ----
hydro_lines_all     = os.path.join(wrk_gdb, "hydro_lines_all")
hydro_lines_prj     = os.path.join(wrk_gdb, "hydro_lines_prj")
hydro_lines_clip    = os.path.join(wrk_gdb, "hydro_lines_clip")
hydro_line_pts      = os.path.join(wrk_gdb, "hydro_line_pts")
hydro_line_pts_ok   = os.path.join(wrk_gdb, "hydro_line_pts_ok")

hydro_polys_all     = os.path.join(wrk_gdb, "hydro_polys_all")
hydro_polys_prj     = os.path.join(wrk_gdb, "hydro_polys_prj")
hydro_polys_clip    = os.path.join(wrk_gdb, "hydro_polys_clip")
hydro_poly_pts      = os.path.join(wrk_gdb, "hydro_poly_pts")
hydro_poly_pts_ok   = os.path.join(wrk_gdb, "hydro_poly_pts_ok")

hydro_pts_merged    = os.path.join(wrk_gdb, "hydro_pts_merged")
hydro_pts_valid     = os.path.join(wrk_gdb, "hydro_pts_valid")
hydro_pts_reduced   = os.path.join(wrk_gdb, "hydro_pts_reduced")

# ---- merged control points ----
surface_controls_all = os.path.join(wrk_gdb, "surface_controls_all")
combined_pts         = os.path.join(wrk_gdb, "combined_points")

# ---- raster outputs ----
initial_ebk_tif       = os.path.join(out_dir, "initial_wt_ebk_5000m.tif")
above_dem_mask_tif    = os.path.join(out_dir, "above_dem_mask_5000m.tif")
above_dem_mask_use_tif= os.path.join(out_dir, "above_dem_mask_use.tif")
starting_heads_tif    = os.path.join(out_dir, "starting_heads_ebk_1000m.tif")

# =====================================================================
# HELPERS
# =====================================================================
def stamp(msg, t0):
    """Print elapsed time in minutes for a major workflow step."""
    print(f"{msg}: {round((time.time() - t0)/60, 2)} min")


def field_exists(fc, field_name):
    """Return True if a field exists in a feature class/table."""
    return field_name in [f.name for f in arcpy.ListFields(fc)]


def add_field_if_needed(fc, field_name, field_type, field_length=None):
    """Add a field only if it does not already exist."""
    if not field_exists(fc, field_name):
        if field_type.upper() == "TEXT":
            arcpy.management.AddField(fc, field_name, field_type, field_length=field_length or 50)
        else:
            arcpy.management.AddField(fc, field_name, field_type)


def print_count(label, fc):
    """Print number of rows in a feature class."""
    print(f"{label}: {arcpy.management.GetCount(fc)[0]}")


def merge_if_any(inputs, out_fc):
    """
    Merge only the inputs that actually exist.
    Returns:
        out_fc if at least one input exists
        None   if no valid inputs exist
    """
    existing = [fc for fc in inputs if fc and arcpy.Exists(fc)]
    if not existing:
        return None

    if arcpy.Exists(out_fc):
        arcpy.management.Delete(out_fc)

    if len(existing) == 1:
        arcpy.management.CopyFeatures(existing[0], out_fc)
    else:
        arcpy.management.Merge(existing, out_fc)

    return out_fc


def project_if_needed(in_fc, out_fc, target_sr):
    """
    Reproject a feature class if its CRS differs from the target CRS.
    Otherwise just copy the feature class.
    """
    if arcpy.Exists(out_fc):
        arcpy.management.Delete(out_fc)

    sr = arcpy.Describe(in_fc).spatialReference
    same = (sr.name == target_sr.name and sr.factoryCode == target_sr.factoryCode)

    if same:
        arcpy.management.CopyFeatures(in_fc, out_fc)
    else:
        tx = arcpy.ListTransformations(sr, target_sr)
        if tx:
            arcpy.management.Project(in_fc, out_fc, target_sr, tx[0])
        else:
            arcpy.management.Project(in_fc, out_fc, target_sr)


def reduce_points_to_model_cells(in_points, value_field, out_points, model_raster, work_gdb, prefix):
    """
    Reduce many points to ONE representative point per model cell.

    Why this helps:
    - prevents many wells in the same cell from overweighting interpolation
    - reduces the number of points going into EBK
    - usually improves speed a lot

    Method:
    1) copy points
    2) compute POINT_X / POINT_Y
    3) compute ROW / COL / CELL_ID from the model raster
    4) average value_field and XY inside each cell
    5) create one output point per cell
    """
    tmp_pts   = os.path.join(work_gdb, f"{prefix}_tmp")
    tmp_stats = os.path.join(work_gdb, f"{prefix}_stats")

    for p in [tmp_pts, tmp_stats, out_points]:
        if arcpy.Exists(p):
            arcpy.management.Delete(p)

    arcpy.management.CopyFeatures(in_points, tmp_pts)

    # Add XY coordinates if they do not already exist
    if "POINT_X" not in [f.name for f in arcpy.ListFields(tmp_pts)]:
        arcpy.management.AddXY(tmp_pts)

    # Read model grid information from the DEM / aligned model raster
    r = arcpy.Raster(model_raster)
    desc = arcpy.Describe(model_raster)
    ext = desc.extent

    xmin = ext.XMin
    ymax = ext.YMax
    dx = float(r.meanCellWidth)
    dy = float(r.meanCellHeight)
    ncol = int(r.width)
    nrow = int(r.height)

    # Add helper fields
    for fld, ftype in [("COL", "LONG"), ("ROW", "LONG"), ("CELL_ID", "TEXT")]:
        if fld not in [f.name for f in arcpy.ListFields(tmp_pts)]:
            if ftype == "TEXT":
                arcpy.management.AddField(tmp_pts, "CELL_ID", "TEXT", field_length=40)
            else:
                arcpy.management.AddField(tmp_pts, fld, ftype)

    # Column index from X coordinate
    arcpy.management.CalculateField(
        tmp_pts,
        "COL",
        f"int((!POINT_X! - {xmin}) / {dx})",
        "PYTHON3"
    )

    # Row index from Y coordinate
    arcpy.management.CalculateField(
        tmp_pts,
        "ROW",
        f"int(({ymax} - !POINT_Y!) / {dy})",
        "PYTHON3"
    )

    # Unique cell identifier
    arcpy.management.CalculateField(
        tmp_pts,
        "CELL_ID",
        'str(!ROW!) + "_" + str(!COL!)',
        "PYTHON3"
    )

    # Keep only points that fall inside the raster bounds and have a value
    where = f"ROW >= 0 AND ROW < {nrow} AND COL >= 0 AND COL < {ncol} AND {value_field} IS NOT NULL"
    lyr = f"{prefix}_lyr"
    arcpy.management.MakeFeatureLayer(tmp_pts, lyr, where)

    # Compute mean value and mean XY for each cell
    arcpy.analysis.Statistics(
        in_table=lyr,
        out_table=tmp_stats,
        statistics_fields=[
            [value_field, "MEAN"],
            ["POINT_X", "MEAN"],
            ["POINT_Y", "MEAN"]
        ],
        case_field="CELL_ID"
    )

    # Convert grouped results back into points
    arcpy.management.XYTableToPoint(
        in_table=tmp_stats,
        out_feature_class=out_points,
        x_field="MEAN_POINT_X",
        y_field="MEAN_POINT_Y",
        coordinate_system=arcpy.Describe(model_raster).spatialReference
    )

    # Create final WT_ELEV field
    add_field_if_needed(out_points, "WT_ELEV", "DOUBLE")
    mean_field = f"MEAN_{value_field}"

    with arcpy.da.UpdateCursor(out_points, [mean_field, "WT_ELEV"]) as cur:
        for row in cur:
            row[1] = row[0]
            cur.updateRow(row)

    print_count(f"Reduced points ({prefix})", out_points)
    return out_points


def run_ebk(
    in_points,
    z_field,
    out_raster,
    cell_size,
    transformation,
    max_local_points,
    overlap_factor,
    num_sim,
    radius_m,
    max_neighbors,
    min_neighbors,
    sector_type,
    semivariogram
):
    n = int(arcpy.management.GetCount(in_points)[0])
    if n < max(min_neighbors, 5):
        raise RuntimeError(f"Not enough points for EBK in {in_points}. Count = {n}")

    if max_local_points < 30 or max_local_points > 10000:
        raise ValueError(f"max_local_points must be between 30 and 10000. Got {max_local_points}")

    if num_sim < 30 or num_sim > 10000:
        raise ValueError(f"num_sim must be between 30 and 10000. Got {num_sim}")

    if overlap_factor <= 0:
        raise ValueError(f"overlap_factor must be > 0. Got {overlap_factor}")

    search_nbhd = arcpy.SearchNeighborhoodStandardCircular(
        radius_m,
        0,
        max_neighbors,
        min_neighbors,
        sector_type
    )

    if arcpy.Exists(out_raster):
        arcpy.management.Delete(out_raster)

    arcpy.ga.EmpiricalBayesianKriging(
        in_points,
        z_field,
        "",
        out_raster,
        cell_size,
        transformation,
        max_local_points,
        overlap_factor,
        num_sim,
        search_nbhd,
        "PREDICTION",
        "",
        "",
        "",
        semivariogram
    )

    print(f"Saved EBK raster: {out_raster}")
    return out_raster

# =====================================================================
# STEP 0 - BASIC ENVIRONMENT
# =====================================================================
# We do NOT set extent to the full DEM.
# Instead, after projecting the boundary, we will set:
#   extent = boundary
#   mask   = boundary
#
# This is a major optimization because it prevents raster operations and
# EBK output from covering a much larger area than needed.
# =====================================================================
t = time.time()

target_sr = arcpy.Describe(dem_tif).spatialReference
arcpy.env.outputCoordinateSystem = target_sr
arcpy.env.snapRaster = dem_tif
arcpy.env.cellSize = FINAL_CELL_SIZE
arcpy.env.parallelProcessingFactor = "50%"

print("Target CRS:", target_sr.name, "|", target_sr.factoryCode)
stamp("Environment initialized", t)


# =====================================================================
# STEP 1 - PREPARE BOUNDARY AND WELLS
# =====================================================================
# Workflow:
#   1) project boundary to model CRS
#   2) set extent/mask to boundary
#   3) project wells
#   4) clip wells to boundary
#   5) filter valid SWL values
#   6) extract DEM elevation to wells
#   7) compute WT_ELEV
# =====================================================================
t = time.time()

project_if_needed(boundary_p, boundary_prj, target_sr)

# Important optimization: process only inside the boundary
arcpy.env.extent = boundary_prj
arcpy.env.mask   = boundary_prj

project_if_needed(Well_path, wells_prj, target_sr)
arcpy.analysis.PairwiseClip(wells_prj, boundary_prj, wells_clip)
print_count("Wells clipped to boundary", wells_clip)

where = f"{SWL_FIELD} IS NOT NULL AND {SWL_FIELD} >= {SWL_MIN} AND {SWL_FIELD} <= {SWL_MAX}"
arcpy.management.MakeFeatureLayer(wells_clip, "wells_lyr", where)
arcpy.management.CopyFeatures("wells_lyr", wells_valid)
print_count("Wells with valid SWL", wells_valid)

# Sample DEM elevation at the well points
ExtractMultiValuesToPoints(wells_valid, [[dem_tif, "DEM_ELEV"]], "NONE")

# Compute the water-table elevation field to interpolate
add_field_if_needed(wells_valid, "WT_ELEV", "DOUBLE")

with arcpy.da.UpdateCursor(wells_valid, [SWL_FIELD, "DEM_ELEV", "WT_ELEV"]) as cur:
    for swl, dem_elev, wt in cur:
        if swl is None:
            cur.updateRow((swl, dem_elev, None))
            continue

        if SWL_MODE == "ELEVATION":
            wt_val = float(swl)

        elif SWL_MODE == "DEPTH_BELOW_GROUND":
            wt_val = None if dem_elev is None else float(dem_elev) - float(swl)

        else:
            raise ValueError("SWL_MODE must be 'ELEVATION' or 'DEPTH_BELOW_GROUND'")

        cur.updateRow((swl, dem_elev, wt_val))

# Keep only points with valid WT_ELEV
arcpy.management.MakeFeatureLayer(wells_valid, "wells_wt_lyr", "WT_ELEV IS NOT NULL")
arcpy.management.CopyFeatures("wells_wt_lyr", wells_wt)
print_count("Wells with WT_ELEV", wells_wt)

stamp("Prepared wells", t)


# =====================================================================
# STEP 2 - REDUCE WELLS TO ONE POINT PER MODEL CELL
# =====================================================================
# This is one of the biggest practical speedups before EBK.
# =====================================================================
t = time.time()

reduce_points_to_model_cells(
    in_points=wells_wt,
    value_field="WT_ELEV",
    out_points=wells_reduced,
    model_raster=dem_tif,
    work_gdb=wrk_gdb,
    prefix="wells"
)

stamp("Reduced wells to one point per model cell", t)


# =====================================================================
# STEP 3 - INITIAL EBK (COARSE / FAST)
# =====================================================================
# This first kriging pass is only used to identify where WT > DEM.
# Because it is a screening raster, we run it at a coarser resolution.
# =====================================================================
t = time.time()

run_ebk(
    in_points=wells_reduced,
    z_field="WT_ELEV",
    out_raster=initial_ebk_tif,
    cell_size=INITIAL_CELL_SIZE,
    transformation=INITIAL_EBK_TRANSFORMATION,
    max_local_points=INITIAL_EBK_MAX_LOCAL_POINTS,
    overlap_factor=INITIAL_EBK_OVERLAP_FACTOR,
    num_sim=INITIAL_EBK_NUM_SIM,
    radius_m=INITIAL_EBK_RADIUS_M,
    max_neighbors=INITIAL_EBK_MAX_NEIGHBORS,
    min_neighbors=INITIAL_EBK_MIN_NEIGHBORS,
    sector_type=INITIAL_EBK_SECTOR_TYPE,
    semivariogram=INITIAL_EBK_SEMIVARIOGRAM
)

stamp("Initial coarse EBK complete", t)


# =====================================================================
# STEP 4 - CREATE CONTROL POINTS WHERE INITIAL WT > DEM
# =====================================================================
# Method:
#   1) compare the initial kriged water table to the DEM
#   2) create a binary mask where WT > DEM
#   3) optionally aggregate that mask to reduce point count
#   4) convert the mask cells to points
#   5) assign those points WT_ELEV = DEM_ELEV
#
# Why:
# These are places where the initial groundwater surface is above land
# surface. The final kriging uses these control points to pull the result
# back toward realistic land-surface elevations in those areas.
# =====================================================================
t = time.time()

# Temporarily switch environment to the coarse initial raster scale
# so the WT > DEM mask is also built at a coarse scale.
old_snap   = arcpy.env.snapRaster
old_cell   = arcpy.env.cellSize
old_extent = arcpy.env.extent
old_mask   = arcpy.env.mask

arcpy.env.snapRaster = initial_ebk_tif
arcpy.env.cellSize   = INITIAL_CELL_SIZE
arcpy.env.extent     = boundary_prj
arcpy.env.mask       = boundary_prj

wt0 = Raster(initial_ebk_tif)
dem = Raster(dem_tif)

above_dem_mask = Con((~IsNull(wt0)) & (~IsNull(dem)) & (wt0 > dem), 1)
above_dem_mask.save(above_dem_mask_tif)
print("Saved coarse WT > DEM mask:", above_dem_mask_tif)

# Optional additional thinning of the mask before RasterToPoint
if MASK_AGG_FACTOR > 1:
    mask_use = Aggregate(
        above_dem_mask,
        MASK_AGG_FACTOR,
        "MAXIMUM",
        "EXPAND",
        "DATA"
    )
    mask_use.save(above_dem_mask_use_tif)
    print("Saved aggregated mask for point creation:", above_dem_mask_use_tif)
else:
    arcpy.management.CopyRaster(above_dem_mask_tif, above_dem_mask_use_tif)
    print("Using unaggregated mask for point creation:", above_dem_mask_use_tif)

# Restore the model-grid environment for later steps
arcpy.env.snapRaster = dem_tif
arcpy.env.cellSize   = FINAL_CELL_SIZE
arcpy.env.extent     = boundary_prj
arcpy.env.mask       = boundary_prj

# Convert WT > DEM cells to points
if arcpy.Exists(surface_pts_raw):
    arcpy.management.Delete(surface_pts_raw)

arcpy.conversion.RasterToPoint(above_dem_mask_use_tif, surface_pts_raw, "Value")
print_count("Surface exceedance points", surface_pts_raw)

# Sample DEM elevation and initial EBK value at these points for QA
ExtractMultiValuesToPoints(
    surface_pts_raw,
    [
        [dem_tif, "DEM_ELEV"],
        [initial_ebk_tif, "WT0_ELEV"]
    ],
    "NONE"
)

# Force these points to the land surface elevation
add_field_if_needed(surface_pts_raw, "WT_ELEV", "DOUBLE")
with arcpy.da.UpdateCursor(surface_pts_raw, ["DEM_ELEV", "WT_ELEV"]) as cur:
    for dem_elev, wt in cur:
        wt_val = None if dem_elev is None else float(dem_elev)
        cur.updateRow((dem_elev, wt_val))

arcpy.management.MakeFeatureLayer(surface_pts_raw, "surface_pts_lyr", "WT_ELEV IS NOT NULL")
arcpy.management.CopyFeatures("surface_pts_lyr", surface_pts_valid)
print_count("Valid surface control points", surface_pts_valid)

# Reduce to one point per final model cell
reduce_points_to_model_cells(
    in_points=surface_pts_valid,
    value_field="WT_ELEV",
    out_points=surface_pts_reduced,
    model_raster=dem_tif,
    work_gdb=wrk_gdb,
    prefix="surface"
)

stamp("Built WT > DEM control points", t)


# =====================================================================
# STEP 5 - OPTIONAL HYDRO CONTROL POINTS
# =====================================================================
# These are extra control points from streams/lakes/wetlands.
#
# For line features:
#   - generate points along lines
#
# For polygon features:
#   - use one inside point per polygon
#
# Then:
#   - sample DEM elevation
#   - optionally keep only points inside the WT > DEM mask
#   - assign WT_ELEV = DEM_ELEV
#   - reduce to one point per model cell
# =====================================================================
t = time.time()

hydro_point_parts = []

# ---------------------------------------------------------------------
# 5A - Line hydro features
# ---------------------------------------------------------------------
if hydro_line_fcs:
    merge_if_any(hydro_line_fcs, hydro_lines_all)
    project_if_needed(hydro_lines_all, hydro_lines_prj, target_sr)
    arcpy.analysis.PairwiseClip(hydro_lines_prj, boundary_prj, hydro_lines_clip)
    print_count("Hydro lines in boundary", hydro_lines_clip)

    if int(arcpy.management.GetCount(hydro_lines_clip)[0]) > 0:
        arcpy.management.GeneratePointsAlongLines(
            Input_Features=hydro_lines_clip,
            Output_Feature_Class=hydro_line_pts,
            Point_Placement="DISTANCE",
            Distance=f"{HYDRO_POINT_SPACING_M} Meters",
            Include_End_Points="END_POINTS"
        )
        print_count("Hydro line points", hydro_line_pts)

        sample_list = [[dem_tif, "DEM_ELEV"]]
        if FILTER_HYDRO_TO_MASK:
            sample_list.append([above_dem_mask_use_tif, "MASK_VAL"])

        ExtractMultiValuesToPoints(hydro_line_pts, sample_list, "NONE")

        where = "DEM_ELEV IS NOT NULL"
        if FILTER_HYDRO_TO_MASK:
            where += " AND MASK_VAL = 1"

        arcpy.management.MakeFeatureLayer(hydro_line_pts, "hydro_line_pts_lyr", where)
        arcpy.management.CopyFeatures("hydro_line_pts_lyr", hydro_line_pts_ok)
        print_count("Hydro line points kept", hydro_line_pts_ok)

        hydro_point_parts.append(hydro_line_pts_ok)

# ---------------------------------------------------------------------
# 5B - Polygon hydro features
# ---------------------------------------------------------------------
if hydro_poly_fcs:
    merge_if_any(hydro_poly_fcs, hydro_polys_all)
    project_if_needed(hydro_polys_all, hydro_polys_prj, target_sr)
    arcpy.analysis.PairwiseClip(hydro_polys_prj, boundary_prj, hydro_polys_clip)
    print_count("Hydro polygons in boundary", hydro_polys_clip)

    if int(arcpy.management.GetCount(hydro_polys_clip)[0]) > 0:
        arcpy.management.FeatureToPoint(hydro_polys_clip, hydro_poly_pts, "INSIDE")
        print_count("Hydro polygon points", hydro_poly_pts)

        sample_list = [[dem_tif, "DEM_ELEV"]]
        if FILTER_HYDRO_TO_MASK:
            sample_list.append([above_dem_mask_use_tif, "MASK_VAL"])

        ExtractMultiValuesToPoints(hydro_poly_pts, sample_list, "NONE")

        where = "DEM_ELEV IS NOT NULL"
        if FILTER_HYDRO_TO_MASK:
            where += " AND MASK_VAL = 1"

        arcpy.management.MakeFeatureLayer(hydro_poly_pts, "hydro_poly_pts_lyr", where)
        arcpy.management.CopyFeatures("hydro_poly_pts_lyr", hydro_poly_pts_ok)
        print_count("Hydro polygon points kept", hydro_poly_pts_ok)

        hydro_point_parts.append(hydro_poly_pts_ok)

# Merge optional hydro control points and assign values
if hydro_point_parts:
    merge_if_any(hydro_point_parts, hydro_pts_merged)
    print_count("Merged hydro points", hydro_pts_merged)

    add_field_if_needed(hydro_pts_merged, "WT_ELEV", "DOUBLE")
    with arcpy.da.UpdateCursor(hydro_pts_merged, ["DEM_ELEV", "WT_ELEV"]) as cur:
        for dem_elev, wt in cur:
            wt_val = None if dem_elev is None else float(dem_elev)
            cur.updateRow((dem_elev, wt_val))

    arcpy.management.MakeFeatureLayer(hydro_pts_merged, "hydro_pts_valid_lyr", "WT_ELEV IS NOT NULL")
    arcpy.management.CopyFeatures("hydro_pts_valid_lyr", hydro_pts_valid)
    print_count("Hydro points with WT_ELEV", hydro_pts_valid)

    reduce_points_to_model_cells(
        in_points=hydro_pts_valid,
        value_field="WT_ELEV",
        out_points=hydro_pts_reduced,
        model_raster=dem_tif,
        work_gdb=wrk_gdb,
        prefix="hydro"
    )
else:
    print("No optional hydro control points were created.")

stamp("Optional hydro control points complete", t)


# =====================================================================
# STEP 6 - MERGE ALL SURFACE-BASED CONTROL POINTS
# =====================================================================
# This step merges:
#   - points created from WT > DEM areas
#   - optional hydro control points
# =====================================================================
t = time.time()

surface_parts = []
if arcpy.Exists(surface_pts_reduced):
    surface_parts.append(surface_pts_reduced)
if arcpy.Exists(hydro_pts_reduced):
    surface_parts.append(hydro_pts_reduced)

if surface_parts:
    if arcpy.Exists(surface_controls_all):
        arcpy.management.Delete(surface_controls_all)

    if len(surface_parts) == 1:
        arcpy.management.CopyFeatures(surface_parts[0], surface_controls_all)
    else:
        arcpy.management.Merge(surface_parts, surface_controls_all)

    print_count("All surface control points", surface_controls_all)
else:
    surface_controls_all = None
    print("No surface control points beyond wells were created.")

stamp("Merged surface control points", t)


# =====================================================================
# STEP 7 - COMBINE WELL POINTS + SURFACE CONTROL POINTS
# =====================================================================
# This follows your workflow:
#   "combined dataset of the water well subset and points representing
#    surface hydrological features is combined and interpolated"
# =====================================================================
t = time.time()

if arcpy.Exists(combined_pts):
    arcpy.management.Delete(combined_pts)

if surface_controls_all and arcpy.Exists(surface_controls_all):
    arcpy.management.Merge([wells_reduced, surface_controls_all], combined_pts)
else:
    arcpy.management.CopyFeatures(wells_reduced, combined_pts)

print_count("Combined interpolation points", combined_pts)
stamp("Combined all interpolation control points", t)


# =====================================================================
# STEP 8 - FINAL EBK STARTING-HEAD SURFACE
# =====================================================================
# This final interpolation uses:
#   - groundwater wells
#   - points from areas where initial WT > DEM
#   - optional surface hydrologic control points
#
# The output is the final gridded estimate of starting heads.
# =====================================================================
t = time.time()

run_ebk(
    in_points=combined_pts,
    z_field="WT_ELEV",
    out_raster=starting_heads_tif,
    cell_size=FINAL_CELL_SIZE,
    transformation=FINAL_EBK_TRANSFORMATION,
    max_local_points=FINAL_EBK_MAX_LOCAL_POINTS,
    overlap_factor=FINAL_EBK_OVERLAP_FACTOR,
    num_sim=FINAL_EBK_NUM_SIM,
    radius_m=FINAL_EBK_RADIUS_M,
    max_neighbors=FINAL_EBK_MAX_NEIGHBORS,
    min_neighbors=FINAL_EBK_MIN_NEIGHBORS,
    sector_type=FINAL_EBK_SECTOR_TYPE,
    semivariogram=FINAL_EBK_SEMIVARIOGRAM
)

stamp("Final EBK starting heads complete", t)


# =====================================================================
# STEP 9 - FINISH
# =====================================================================
print("\nDone.")
print("Work GDB:", wrk_gdb)
print("Initial EBK raster:", initial_ebk_tif)
print("WT > DEM mask raster:", above_dem_mask_tif)
print("Mask used for point creation:", above_dem_mask_use_tif)
print("Final starting-head raster:", starting_heads_tif)

# Check licenses back in
arcpy.CheckInExtension("Spatial")
arcpy.CheckInExtension("GeoStats")

## 10. Optional / archived alternative workflows
These cells were kept for reference because they represent alternative or earlier versions of parts of the workflow.


### Alternative CHD workflow from existing idomain and water mask

This is a more direct shoreline-CHD workflow that starts from an existing idomain raster and a water-mask raster. It is useful as an alternative if the earlier CHD point workflow is not being used.


In [ ]:
import arcpy, os
import numpy as np

arcpy.env.overwriteOutput = True

# --- INPUTS ---
idomain_ras    = r"D:\path\to\idomain_or_ibound.tif"  # active > 0
water_mask_ras = r"D:\path\to\domain_water_mask.tif"  # lakes = -1
out_dir        = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\CHD"
os.makedirs(out_dir, exist_ok=True)

out_fc = os.path.join(out_dir, "CHD_shoreline_cells.shp")

# --- lake stage (meters) ---
# Use a constant for now. Replace with per-lake or time-varying values later.
LAKE_STAGE_M = 176.0

# -----------------------------
# Read rasters to numpy
# -----------------------------
id_r = arcpy.Raster(idomain_ras)
wm_r = arcpy.Raster(water_mask_ras)

# Alignment check (important)
if (id_r.meanCellWidth != wm_r.meanCellWidth) or (id_r.meanCellHeight != wm_r.meanCellHeight):
    raise RuntimeError("Cellsize mismatch between idomain and water mask.")
if (id_r.extent.XMin != wm_r.extent.XMin) or (id_r.extent.YMax != wm_r.extent.YMax) or \
   (id_r.width != wm_r.width) or (id_r.height != wm_r.height):
    raise RuntimeError("Grid mismatch (extent/shape) between idomain and water mask. They must be aligned.")

id_arr = arcpy.RasterToNumPyArray(id_r, nodata_to_value=0)
wm_arr = arcpy.RasterToNumPyArray(wm_r, nodata_to_value=0)

active = id_arr > 0
lake   = wm_arr == -1

# -----------------------------
# Find active cells adjacent to lake (4-neighbor)
# (no wraparound; uses slicing)
# -----------------------------
adj = np.zeros_like(active, dtype=bool)

# lake north of cell (row-1)
adj[1:,  :] |= active[1:,  :] & lake[:-1, :]
# lake south of cell (row+1)
adj[:-1, :] |= active[:-1, :] & lake[1:,  :]
# lake west of cell (col-1)
adj[:, 1:]   |= active[:, 1:] & lake[:, :-1]
# lake east of cell (col+1)
adj[:, :-1]  |= active[:, :-1] & lake[:, 1:]

rows, cols = np.where(adj)
print("CHD shoreline cells:", len(rows))

# -----------------------------
# Export points shapefile (cell centers)
# -----------------------------
sr = id_r.spatialReference
cellw = float(id_r.meanCellWidth)
cellh = float(id_r.meanCellHeight)
xmin  = float(id_r.extent.XMin)
ymax  = float(id_r.extent.YMax)

if arcpy.Exists(out_fc):
    arcpy.management.Delete(out_fc)

arcpy.management.CreateFeatureclass(out_dir, os.path.basename(out_fc), "POINT", spatial_reference=sr)
arcpy.management.AddField(out_fc, "row", "LONG")
arcpy.management.AddField(out_fc, "col", "LONG")
arcpy.management.AddField(out_fc, "head_m", "DOUBLE")

with arcpy.da.InsertCursor(out_fc, ["SHAPE@XY", "row", "col", "head_m"]) as icur:
    for r, c in zip(rows, cols):
        # RasterToNumPyArray starts at upper-left; row increases downward
        x = xmin + (c + 0.5) * cellw
        y = ymax - (r + 0.5) * cellh
        icur.insertRow(((x, y), int(r), int(c), float(LAKE_STAGE_M)))

print("Wrote:", out_fc)


### Earlier bathymetry merge attempt (kept for reference)

This is an earlier version of the lake-bathymetry merge workflow. I kept it in a reference section because the later standardized merge cell is more complete.


In [ ]:
import arcpy
import os

arcpy.env.overwriteOutput = True

# ---------------------------------------------------
# INPUTS
# ---------------------------------------------------
sup_bathy = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\GreatLakes_bathymetry\out_contour_LakeSuperior.shp"
Michigan_bathy = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\GreatLakes_bathymetry\Lake_Michigan_Contours.shp"
Huron_bathy = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\GreatLakes_bathymetry\Lake_Huron_Contours.shp"
Erie_bathy = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\GreatLakes_bathymetry\Lake_Erie_Contours.shp"
Ontario_bathy = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\GreatLakes_bathymetry\Lake_Ontario_Contours.shp"

out_merged = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\GreatLakes_bathymetry\GreatLakes_bathymetry_projected_merge.shp"

inputs = {
    "sup": sup_bathy,
    "mic": Michigan_bathy,
    "hur": Huron_bathy,
    "eri": Erie_bathy,
    "ont": Ontario_bathy,
}

# ---------------------------------------------------
# CREATE A TEMP FILE GDB
# ---------------------------------------------------
work_dir = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\GreatLakes_bathymetry"
temp_gdb = os.path.join(work_dir, "temp_bathy_merge.gdb")

if arcpy.Exists(temp_gdb):
    arcpy.management.Delete(temp_gdb)

arcpy.management.CreateFileGDB(work_dir, "temp_bathy_merge.gdb")
print("Created temp GDB:", temp_gdb)

# ---------------------------------------------------
# TARGET CRS = SUPERIOR
# ---------------------------------------------------
target_sr = arcpy.Describe(sup_bathy).spatialReference
print("Target CRS:", target_sr.name, "| factoryCode:", target_sr.factoryCode)

projected_inputs = []

# ---------------------------------------------------
# PROCESS EACH INPUT
# ---------------------------------------------------
for key, fc in inputs.items():
    print("\n===================================================")
    print("Processing:", key, "->", os.path.basename(fc))

    if not arcpy.Exists(fc):
        raise FileNotFoundError(f"Missing input: {fc}")

    desc = arcpy.Describe(fc)
    sr = desc.spatialReference
    count0 = int(arcpy.management.GetCount(fc)[0])

    print("Original count:", count0)
    print("Original CRS:", sr.name, "| factoryCode:", sr.factoryCode)

    # -------------------------
    # 1) copy to temp gdb
    # -------------------------
    fc_copy = os.path.join(temp_gdb, f"{key}_copy")
    if arcpy.Exists(fc_copy):
        arcpy.management.Delete(fc_copy)

    try:
        arcpy.management.CopyFeatures(fc, fc_copy)
    except Exception as e:
        print("CopyFeatures failed for:", fc)
        raise e

    if not arcpy.Exists(fc_copy):
        raise RuntimeError(f"CopyFeatures did not create: {fc_copy}")

    print("Copied to:", fc_copy)
    print("Count after copy:", arcpy.management.GetCount(fc_copy)[0])

    # -------------------------
    # 2) repair geometry
    # -------------------------
    try:
        arcpy.management.RepairGeometry(fc_copy)
        print("RepairGeometry: done")
    except Exception as e:
        print("RepairGeometry failed:", e)

    if not arcpy.Exists(fc_copy):
        raise RuntimeError(f"Feature class disappeared after RepairGeometry: {fc_copy}")

    print("Count after repair:", arcpy.management.GetCount(fc_copy)[0])

    # -------------------------
    # 3) project if needed
    # -------------------------
    fc_proj = os.path.join(temp_gdb, f"{key}_proj")
    if arcpy.Exists(fc_proj):
        arcpy.management.Delete(fc_proj)

    same_crs = (
        sr.factoryCode == target_sr.factoryCode
        and sr.name == target_sr.name
    )

    if same_crs:
        print("Already in target CRS.")
        projected_inputs.append(fc_copy)
    else:
        tx = arcpy.ListTransformations(sr, target_sr)
        transform = tx[0] if tx else None

        if transform:
            print("Using transformation:", transform)
            arcpy.management.Project(fc_copy, fc_proj, target_sr, transform)
        else:
            print("No transformation returned; projecting without explicit transform.")
            arcpy.management.Project(fc_copy, fc_proj, target_sr)

        if not arcpy.Exists(fc_proj):
            raise RuntimeError(f"Project did not create: {fc_proj}")

        print("Projected to:", fc_proj)
        print("Count after project:", arcpy.management.GetCount(fc_proj)[0])
        projected_inputs.append(fc_proj)

# ---------------------------------------------------
# MERGE PROJECTED INPUTS
# ---------------------------------------------------
if arcpy.Exists(out_merged):
    arcpy.management.Delete(out_merged)

print("\n===================================================")
print("Merging projected inputs...")
for fc in projected_inputs:
    print(" ", fc, "| count:", arcpy.management.GetCount(fc)[0])

arcpy.management.Merge(projected_inputs, out_merged)

print("\nMerged output saved to:")
print(out_merged)
print("Merged count:", arcpy.management.GetCount(out_merged)[0])
print("Merged CRS:", arcpy.Describe(out_merged).spatialReference.name)

# ---------------------------------------------------
# OPTIONAL: ADD LAKE NAME FIELD
# ---------------------------------------------------
lake_field = "LAKE_NAME"
field_names = [f.name for f in arcpy.ListFields(out_merged)]
if lake_field not in field_names:
    arcpy.management.AddField(out_merged, lake_field, "TEXT", field_length=20)

# fill lake name by appending from source pieces would need done before merge;
# so this section is just a reminder if you want provenance later.

print("\nDone.")

## End of organized notebook

This version is meant to be easier to read and follow than the original notebook.  
A good next step would be to turn repeated hard-coded paths into a single **configuration cell** at the top so the whole notebook can be rerun with fewer edits.
